
# colab_19 — scGPT continued pretraining (CPT), **per-study regime** · N=1

The third and last CPT regime for scGPT, mirroring what colab_14 did for Geneformer. Instead of one
adapter trained on the concatenated + shuffled train split (colab_16), this trains **three
independent LoRA adapters from the same frozen base** — one per study (SEA-AD, Li2025, Haney2024) —
each on that study's slice of the *same frozen donor split*. Parallel and independent, not
sequential: no adapter ever sees another study's gradients.

Each checkpoint is then re-embedded over the full glia substrate and passed through detector #1,
exactly as colab_16 did. Evals #1/#2 on the three checkpoints are a **separate downstream notebook**,
the same way colab_16 was training-only and colab_17 carried the evals.

## Why this is not just a re-demonstration

The per-study *mechanic* was already shown end-to-end on Geneformer, so repeating it for its own sake
would buy little. What makes this run worth the calendar time is that it is a near-natural test of
colab_14's own explanation for its headline number.

colab_14 found the three Geneformer checkpoints drifted by materially different amounts —
population-matched `drift_all` of 0.00543 / 0.00300 / 0.00359 (SEA-AD / Li2025 / Haney2024), a
**1.81x spread** — and that the spread rank-ordered with each study's **median tokenized sequence
length** (2852 / 1612 / 2077) rather than with its training-step count (1079 / 660 / 261). Rank on
length is a perfect match; rank on steps is not. That was offered as the candidate explanation, on a
single three-point ordering.

scGPT tests it almost by construction, because **scGPT's context ceiling truncates the very lever
that explanation rests on**. Geneformer's glia sequences run to its full 4096-token ceiling; scGPT
caps at 1200 tokens, and colab_16 measured 86.3% of this substrate as sitting above that cap. For any
cell over the cap, effective input length is 1200 regardless of how many genes it actually detected —
so most of the cross-study length difference is flattened before the model sees anything.

**Pre-registered, written before the run (§3b states it in the notebook, §6b resolves it):**

- If length drove colab_14's spread, the per-study `drift_all` spread here should be **materially
  below 1.81x**, and should not rank-order with pre-cap detected-gene counts.
- If the spread instead survives at roughly 1.8x or more, *with* the same rank order, the length
  attribution was incomplete and something study-specific — gradient content, composition, batch —
  is doing the work.

**This prediction is conditional on a premise the notebook measures rather than assumes.** §3b
computes each study's over-cap fraction and post-cap effective length *before* any GPU work. If some
study's cells largely sit *under* the cap, the lever is not flattened for that study and the
prediction weakens accordingly — that has to be read first, not discovered afterwards.

**Honest limits, stated up front.** N=1 per study, three points, no seeds — this can support or
embarrass the length explanation, it cannot settle it. And epoch matching (below) equalizes per-cell
exposure but **not** per-adapter optimisation budget: SEA-AD still gets 4.1x Haney2024's update
count, exactly as in colab_14. Any per-study difference found here remains confounded with that.

**Outputs:** three LoRA adapters, three full-substrate embeddings, a path-matched re-embedding of
colab_16's aggregated adapter, and a `scgpt_cpt_per_study` block in `audit_report.json`.


## 1 — Setup



### 1a — Run-control flags, Drive, repo, scGPT install, checkpoint

`SMOKE` is the single run-control flag; every output path below is derived from it, so a rehearsal
can waste time but cannot overwrite a real artifact. The subsample is applied in §2c — after the
substrate and split checks have run against the full object, and before the first expensive stage.

scGPT is installed `--no-deps` at the same pinned commit every existing scGPT embedding in this
project was produced under, and `peft` is asserted exactly, because the LoRA merge semantics for
`nn.MultiheadAttention` are version-sensitive and a skew there would change results silently.
flash-attn stays absent, so attention resolves to the PyTorch path — the path colab_10/16/17/18 all
embedded under.

**Restart the runtime after this cell**, every time.


In [1]:

import os, subprocess, sys
from datetime import date
from google.colab import drive

# ------------------------------ the one live switch ------------------------------
SMOKE = False   # plumbing rehearsal on a small subsample; all writes get a _SMOKE suffix
# ---------------------------------------------------------------------------------

SMOKE_CAP = 40             # cells per (study x split x lineage) group when SMOKE
SUFFIX    = "_SMOKE" if SMOKE else ""
RUN_TAG   = "seed0"
TODAY     = date.today().isoformat()

# Re-embedding colab_16's stored aggregated adapter through THIS notebook's embedding path is what
# makes §6b's per-study-vs-aggregated comparison path-matched (see 5a). Costs one extra
# full-substrate pass; turning it off leaves only the stored colab_16 numbers, which were produced
# under a different input encoding.
REEMBED_AGGREGATED = True

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print("Python:", sys.version.split()[0])
print("repo commit:", subprocess.run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"],
                                     capture_output=True, text=True).stdout.strip())

SCGPT_PIN  = "cebd6fae655b9c585a4807daa3ac31bb764f06b4"
MODEL_DIR  = os.path.join(DRIVE_ROOT, "scgpt_whole_human")
CKPT_FILES = ["vocab.json", "args.json", "best_model.pt"]

!pip install --no-deps "git+https://github.com/bowang-lab/scGPT.git@{SCGPT_PIN}"
!pip install -r {REPO_PATH}/requirements_scgpt.txt
# Colab's base image can ship torchao 0.10.0, and peft 0.19.1's `is_torchao_available()` RAISES on
# any version below 0.16.0 instead of returning False -- that kills the adapter reload in 5c.
# Nothing in the scGPT stack imports torchao, so removing it makes the probe return False.
!pip uninstall -y torchao

import peft
assert peft.__version__ == "0.19.1", (
    f"peft {peft.__version__} != 0.19.1 -- this notebook depends on that version's LoRA merge "
    "semantics for nn.MultiheadAttention; a skew changes results without raising")

_have_ckpt = all(os.path.exists(os.path.join(MODEL_DIR, f)) for f in CKPT_FILES)
assert _have_ckpt, (
    f"missing pretrained whole-human checkpoint files in {MODEL_DIR}; colab_10/16 downloaded these")
print("checkpoint present:", MODEL_DIR)

import torch
assert torch.cuda.is_available(), "no GPU -- refusing to run CPT on CPU"
print("GPU:", torch.cuda.get_device_name(0))
print(f"SMOKE={SMOKE} | SUFFIX={SUFFIX!r} | REEMBED_AGGREGATED={REEMBED_AGGREGATED}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.12.13
repo commit: 8399388cc42b5392e3fa07f35b36b243a7b0fdef
  Cloning https://github.com/bowang-lab/scGPT.git (to revision cebd6fae655b9c585a4807daa3ac31bb764f06b4) to /tmp/pip-req-build-tgl_oeub
  Running command git clone --filter=blob:none --quiet https://github.com/bowang-lab/scGPT.git /tmp/pip-req-build-tgl_oeub
  Running command git rev-parse -q --verify 'sha^cebd6fae655b9c585a4807daa3ac31bb764f06b4'
  Running command git fetch -q https://github.com/bowang-lab/scGPT.git cebd6fae655b9c585a4807daa3ac31bb764f06b4
  Resolved https://github.com/bowang-lab/scGPT.git to commit cebd6fae655b9c585a4807daa3ac31bb764f06b4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
checkpoint present: /content/drive/MyDrive/ad-glia-fm-prep/scgpt_whole_human
GPU: NVIDIA A100-SXM4-80G

> **Interpretation — environment installed clean, checkpoint/repo pinned to the values every other scGPT notebook in this project used (1a).**
>
> Drive was already mounted from a prior session cell/tab, so no fresh mount happened. The repo checked out at commit `8399388` -- the reviewed scaffold this run executes. scGPT installed `--no-deps` from the same pinned upstream commit (`cebd6fae`) every scGPT embedding in this project has used, and peft resolved to `0.19.1`, the version this project's LoRA-target-regex recipe (colab_16) and adapter-reload logic depend on for correct `nn.MultiheadAttention` merge behaviour. Every other listed dependency (scanpy 1.12.3, anndata 0.13.2, datasets 5.0.0, typing_extensions >=4.16.0) was already present and satisfied, not freshly installed -- the "Requirement already satisfied" lines are Colab's stock image, not something this cell changed. The `pip uninstall -y torchao` line was a no-op in this session (`Skipping torchao as it is not installed`), so the peft-0.19.1 `is_torchao_available()` hard-raise it exists to pre-empt could not have fired here.

### 1b — pip freeze + env JSON (records the exact stack this run used)


In [2]:

import json, platform

NOTEBOOK_ID = "colab_19"
VER_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VER_DIR, exist_ok=True)

freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
FREEZE_PATH = os.path.join(VER_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze{SUFFIX}.txt")
with open(FREEZE_PATH, "w") as f:
    f.write(freeze)

# A git/HF clone is a third category pip freeze never sees. The pin itself is enforced by the
# versioned install line in cell 3 (`@{SCGPT_PIN}`) -- this only records what actually got resolved,
# for the Methods-section audit trail, not an independent re-check of the SHA.
import scgpt
_scgpt_ver = getattr(scgpt, "__version__", "unknown")

import numpy, scanpy, anndata, sklearn

env = {
    "notebook": NOTEBOOK_ID, "date": TODAY, "smoke": bool(SMOKE),
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "peft": peft.__version__,
    "scgpt_version": _scgpt_ver, "scgpt_commit_pinned": SCGPT_PIN,
    "scanpy_version": scanpy.__version__, "anndata_version": anndata.__version__,
    "numpy_version": numpy.__version__, "sklearn_version": sklearn.__version__,
    "checkpoint_dir": os.path.basename(MODEL_DIR),
}
ENV_PATH = os.path.join(VER_DIR, f"{NOTEBOOK_ID}_{TODAY}_env{SUFFIX}.json")
with open(ENV_PATH, "w") as f:
    json.dump(env, f, indent=2)
print(json.dumps(env, indent=2))
print("\nwrote:", FREEZE_PATH, "\n       ", ENV_PATH)


{
  "notebook": "colab_19",
  "date": "2026-07-28",
  "smoke": false,
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "gpu": "NVIDIA A100-SXM4-80GB",
  "peft": "0.19.1",
  "scgpt_version": "0.2.5",
  "scgpt_commit_pinned": "cebd6fae655b9c585a4807daa3ac31bb764f06b4",
  "scanpy_version": "1.12.3",
  "anndata_version": "0.13.2",
  "numpy_version": "2.4.6",
  "sklearn_version": "1.6.1",
  "checkpoint_dir": "scgpt_whole_human"
}

wrote: /content/ad-glia-fm-prep/outputs/software_versions/colab_19_2026-07-28_pip_freeze.txt 
        /content/ad-glia-fm-prep/outputs/software_versions/colab_19_2026-07-28_env.json


/tmp/ipykernel_13849/2724583678.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  "scanpy_version": scanpy.__version__, "anndata_version": anndata.__version__,
/tmp/ipykernel_13849/2724583678.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  "scanpy_version": scanpy.__version__, "anndata_version": anndata.__version__,


> **Interpretation — environment snapshot recorded for the Methods section; SMOKE confirmed False (1b).**
>
> The printed JSON is the exact stack this run used: SMOKE `false` (a real run, not a rehearsal), Python 3.12.13 on an NVIDIA A100-SXM4-80GB, torch 2.11.0+cu128, CUDA 12.8, peft 0.19.1, scGPT 0.2.5 at the pinned commit `cebd6fae`, scanpy 1.12.3, anndata 0.13.2, numpy 2.4.6, sklearn 1.6.1 -- written to both a pip-freeze text file and this env JSON under `outputs/software_versions/`, per this project's standing rule that every run snapshots its exact dependency stack for the eventual Methods section. The two `FutureWarning`s about `__version__` being deprecated for scanpy/anndata are cosmetic: both packages still return the correct string from the deprecated attribute, so the recorded versions are unaffected -- the cell reads `__version__` directly and has no `importlib.metadata` fallback branch that could have been taken instead.

## 2 — Substrate, schema, frozen split, per-study slices



### 2a — Load both labelled subsets, concatenate, guard raw counts, apply scGPT's input transform

The substrate is rebuilt from the two labelled subsets rather than loaded from a saved file, so
`cell_index` is regenerated deterministically and matches every other notebook that rebuilds it the
same way. `cell_index` is the only key used to realign embeddings later — row order is never trusted
on its own.

The raw-counts guard matters more than it looks: scGPT's input pipeline expects raw counts and
applies `normalize_total(1e4)` + `log1p` itself. Handing it an already-normalized matrix would
double-transform silently and nothing downstream would raise. The transform applied here is
character-for-character the one colab_10 and colab_16 used, because detector #1 compares against
embeddings produced under it and any difference would be indistinguishable from CPT drift.


In [3]:

import gc
import numpy as np, pandas as pd, anndata as ad, scanpy as sc, scipy.sparse as sp

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:28s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

sc.settings.verbosity = 1

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
print("microglia subset:", micro.shape)
print("astrocyte subset:", astro.shape)
assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"

micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()

# Concatenation ORDER is load-bearing: cell_index is assigned by position below, and every stored
# scGPT embedding this notebook aligns against was built micro-then-astro. 5b cross-checks the
# labels row by row, which is what would actually catch a reversal.
glia = ad.concat([micro, astro], axis=0, join="outer", label=None, index_unique=None)
del micro, astro; gc.collect()
glia.obs_names_make_unique()
glia.obs["cell_index"] = np.arange(glia.n_obs)
print("concatenated glia substrate:", glia.shape)
_ram("after concat")

# --- raw-counts guard: scGPT's transform must START from raw counts -----------------------------
rng_guard = np.random.default_rng(0)
probe_idx = rng_guard.choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
probe = glia[probe_idx].X
probe = probe.toarray() if sp.issparse(probe) else np.asarray(probe)
nz = probe[probe != 0]
frac_int = float(np.mean(np.isclose(nz, np.round(nz)))) if nz.size else 1.0
print(f"raw-counts guard: {frac_int:.4f} of sampled nonzero values are integers")
assert frac_int >= 0.99, (
    "substrate does not look like raw counts -- scGPT applies normalize_total+log1p itself and "
    "would double-transform silently")

sc.pp.normalize_total(glia, target_sum=1e4)
sc.pp.log1p(glia)
print("applied normalize_total(1e4) + log1p -- identical to colab_10/16")
_ram("after transform")


microglia subset: (54805, 26514)
astrocyte subset: (87783, 26514)
concatenated glia substrate: (142588, 26514)
[RAM] after concat                :   7.0 / 179.4 GB (5%)
raw-counts guard: 1.0000 of sampled nonzero values are integers
applied normalize_total(1e4) + log1p -- identical to colab_10/16
[RAM] after transform             :   7.3 / 179.4 GB (5%)


> **Interpretation — the frozen 142,588-cell glia substrate reloads at its exact recorded shape, raw counts confirmed before any transform (2a).**
>
> Microglia (54,805 cells) and astrocyte (87,783 cells) subsets concatenate to 142,588 cells across 26,514 genes -- matching the post-CPT-drop substrate every FM notebook in this project loads, rebuilt from the two labelled subset files rather than read from a single cached object so that `cell_index` regenerates deterministically. The raw-counts guard reports 1.0000 -- every sampled nonzero value in `.X` is a whole number -- confirming the input scGPT's `normalize_total(1e4)` + `log1p` transform expects (raw counts, not already-normalized data) is genuinely present; had this guard failed, the transform below would have silently double-normalized already-processed data. RAM after concatenation sits at 7.0 of 179.4 GB (5%), leaving ample headroom for the three separate model instances 4a-4c will build in turn.


### 2b — Schema and substrate reference numbers

Hard-stops against the frozen substrate every FM notebook in this project loads. A mismatch is never
a new substrate — it means something upstream changed and the comparison to colab_16 is void.


In [4]:

REF_N_CELLS  = 142588
REF_LINEAGE  = {"astrocyte": 87783, "microglia": 54805}
REF_STUDIES  = {"SEA-AD", "Li2025", "Haney2024"}

REQUIRED_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "cell_index"]
missing = [c for c in REQUIRED_OBS if c not in glia.obs.columns]
assert not missing, f"substrate is missing required obs columns: {missing}"
for c in REQUIRED_OBS:
    n_null = int(glia.obs[c].isna().sum())
    assert n_null == 0, f"obs['{c}'] has {n_null} nulls -- a null study_id/donor_id would silently misassign a per-study slice"
print("schema complete, zero nulls in:", ", ".join(REQUIRED_OBS))

assert glia.n_obs == REF_N_CELLS, f"substrate {glia.n_obs} cells != frozen reference {REF_N_CELLS}"
lin_counts = glia.obs["lineage"].value_counts().to_dict()
assert {k: int(v) for k, v in lin_counts.items()} == REF_LINEAGE, f"lineage counts {lin_counts} != {REF_LINEAGE}"
studies_here = set(glia.obs["study_id"].astype(str).unique())
assert studies_here == REF_STUDIES, f"studies {studies_here} != {REF_STUDIES}"

STUDIES = ["SEA-AD", "Li2025", "Haney2024"]      # fixed order; every table below uses it
SLUG = {"SEA-AD": "seaad", "Li2025": "li2025", "Haney2024": "haney2024"}

print(f"\nsubstrate verified: {glia.n_obs} cells, {glia.n_vars} genes")
print(glia.obs.groupby(["study_id", "lineage"], observed=True).size().unstack(fill_value=0))


schema complete, zero nulls in: lineage, substate, apoe_carrier, study_id, donor_id, cell_index

substrate verified: 142588 cells, 26514 genes
lineage    astrocyte  microglia
study_id                       
Li2025         26712      20158
SEA-AD         48072      27562
Haney2024      12999       7085


> **Interpretation — schema and per-study/per-lineage cell counts match the frozen substrate exactly; a mismatch here would have been a hard stop (2b).**
>
> All six required obs columns (`lineage`, `substate`, `apoe_carrier`, `study_id`, `donor_id`, `cell_index`) carry zero nulls, and the substrate re-verifies at exactly 142,588 cells / 26,514 genes -- the same numbers 2a produced and the same numbers every FM notebook in this project checks against before proceeding. The study-by-lineage breakdown (SEA-AD 48,072 astro / 27,562 micro; Li2025 26,712 astro / 20,158 micro; Haney2024 12,999 astro / 7,085 micro) is the same per-study composition colab_16's aggregated run loaded -- this notebook's three per-study slices in 2c partition exactly these same cells, study by study, rather than drawing a fresh sample. Per this project's standing rule, any mismatch against these frozen reference numbers would mean something upstream had changed and would be a hard stop, never grounds for silently accepting a new substrate.


### 2c — Frozen split verification, per-study train slices, then the smoke subsample

The donor split is **verified, never redrawn**. `outputs/donor_split.json` was frozen at the first
CPT run and every regime and both FMs have used it since. The per-study train slices are then read
off that same split — they partition the 94,963 training cells, which is what makes §4b's
epoch-matched step budget sum to colab_16's 2000 steps across the three adapters together.

The smoke subsample is applied last, after every check above has run against the full object.


In [5]:

SPLIT_PATH = os.path.join(REPO_PATH, "outputs", "donor_split.json")
assert os.path.exists(SPLIT_PATH), f"missing frozen split {SPLIT_PATH}"
with open(SPLIT_PATH) as f:
    split_artifact = json.load(f)

REF_SEED       = 32
REF_MARGIN     = 10
REF_N_DONORS   = {"train": 101, "val": 22, "test": 22}
REF_N_CELLS_SP = {"train": 94963, "val": 23824, "test": 23801}
REF_N_TRAIN_BY_STUDY = {"SEA-AD": 51218, "Li2025": 31349, "Haney2024": 12396}   # colab_14 3a

assert int(split_artifact["seed"]) == REF_SEED, f"split seed {split_artifact['seed']} != {REF_SEED}"
assert int(split_artifact["test_worst_case_margin"]) == REF_MARGIN, "split test margin != reference"
assert {k: int(v) for k, v in split_artifact["n_donors"].items()} == REF_N_DONORS, "split donor counts != reference"

split_map = split_artifact["donor_split"]
glia.obs["split"] = glia.obs["donor_id"].astype(str).map(split_map)
assert not glia.obs["split"].isna().any(), "some substrate donors are absent from the frozen split"
glia.obs["split"] = glia.obs["split"].astype("category")

cell_counts = glia.obs["split"].value_counts().to_dict()
assert {k: int(v) for k, v in cell_counts.items()} == REF_N_CELLS_SP, (
    f"cells per split {cell_counts} != reference {REF_N_CELLS_SP}")
print(f"frozen split verified: seed {REF_SEED}, margin {REF_MARGIN}, "
      f"{REF_N_DONORS} donors, {REF_N_CELLS_SP} cells")

# --- per-study train slices: these are the denominators the step budget divides by ---------------
train_mask = (glia.obs["split"] == "train").values
n_train_by_study = (glia.obs.loc[train_mask, "study_id"].astype(str)
                    .value_counts().reindex(STUDIES).astype(int).to_dict())
assert n_train_by_study == REF_N_TRAIN_BY_STUDY, (
    f"per-study train counts {n_train_by_study} != colab_14's reference {REF_N_TRAIN_BY_STUDY} -- "
    "the two per-study regimes would not be training on the same slices")
assert sum(n_train_by_study.values()) == REF_N_CELLS_SP["train"], "per-study train slices do not partition the train split"
print("\nper-study train slices (partition the 94,963 train cells):")
for s in STUDIES:
    print(f"  {s:12} n_train {n_train_by_study[s]:6d}")

test_frac = (glia.obs.loc[glia.obs['split'] == 'test', 'study_id']
             .value_counts(normalize=True).reindex(STUDIES))
print("\ntest-split study composition (audit: no study >60%):")
for s in STUDIES:
    print(f"  {s:12} {test_frac[s]:.3f}")
assert test_frac.max() <= 0.60, "a single study exceeds 60% of the test split -- composition artifact"

# --- smoke subsample, applied LAST -------------------------------------------------------------
if SMOKE:
    keys = glia.obs[["study_id", "split", "lineage"]].astype(str).agg("|".join, axis=1)
    rng_smoke = np.random.default_rng(0)
    take = []
    for k, idx in pd.Series(np.arange(glia.n_obs)).groupby(keys.values):
        idx = idx.values
        take.append(rng_smoke.choice(idx, size=min(SMOKE_CAP, len(idx)), replace=False))
    keep = np.sort(np.concatenate(take))
    glia = glia[keep].copy()
    print(f"\n[SMOKE] subsampled to {glia.n_obs} cells "
          f"(<= {SMOKE_CAP} per study x split x lineage). Reference asserts above ran on the FULL object.")
    print(glia.obs.groupby(["study_id", "split"], observed=True).size().unstack(fill_value=0))
_ram("after split + subsample")


frozen split verified: seed 32, margin 10, {'train': 101, 'val': 22, 'test': 22} donors, {'train': 94963, 'val': 23824, 'test': 23801} cells

per-study train slices (partition the 94,963 train cells):
  SEA-AD       n_train  51218
  Li2025       n_train  31349
  Haney2024    n_train  12396

test-split study composition (audit: no study >60%):
  SEA-AD       0.567
  Li2025       0.300
  Haney2024    0.133
[RAM] after split + subsample     :   7.3 / 179.4 GB (5%)


> **Interpretation — the frozen donor split re-verifies exactly, and the three per-study train slices partition all 94,963 training cells with no overlap and no leftover (2c).**
>
> The donor split -- seed 32, margin 10, 101/22/22 donors, 94,963/23,824/23,801 cells -- is read from `outputs/donor_split.json` and re-verified against these exact numbers rather than redrawn, per this project's standing rule that the split is frozen after its first use and every regime and both FMs since have shared it. Splitting the 94,963 training cells by study gives SEA-AD 51,218, Li2025 31,349, and Haney2024 12,396 -- these three numbers sum to exactly 94,963, confirming the per-study slices are a true partition of the training set, not an independent re-sample that could double-count or drop cells. This is the number 4b's epoch-matched step budget is built from: each study's step count is derived from its own `n_train`, so an error in this partition would propagate directly into every training-step calculation that follows. The test-split study composition -- SEA-AD 56.7%, Li2025 30.0%, Haney2024 13.3% -- is unchanged from every other notebook that has verified this split, under the 60% single-study ceiling this project's audit treats as a hard stop, with SEA-AD the closest to it at 3.3 points of margin.

## 3 — Vocabulary and input geometry



### 3a — Vocabulary intersection, APOE gate, in-vocab gene subset

scGPT reads gene **symbols** directly, with no Ensembl mapping step. The steps below reproduce, in
order, what the library's own embedding entry point does: append special tokens if the checkpoint's
vocabulary lacks them, mark each panel gene with its vocabulary id (`-1` when absent), drop the
out-of-vocabulary genes, then build the gene-id array **from the surviving genes in their surviving
order**. That ordering is load-bearing — the id array indexes the same columns the count matrix is
sliced by, and a mismatch would feed the model the wrong gene for every value without raising.

APOE remains a pre-registered hard fail: without it the downstream APOE-axis eval cannot run for
this FM at all. The in-vocabulary fraction is cross-checked against what colab_10 recorded,
recomputed here rather than read back, so it is a real agreement check and not a restatement.


In [6]:

from scgpt.tokenizer.gene_tokenizer import GeneVocab

with open(os.path.join(MODEL_DIR, "args.json")) as f:
    model_configs = json.load(f)
vocab = GeneVocab.from_file(os.path.join(MODEL_DIR, "vocab.json"))

SPECIAL_TOKENS = ["<pad>", "<cls>", "<eoc>"]
for t in SPECIAL_TOKENS:
    if t not in vocab:
        vocab.append_token(t)
vocab.set_default_index(vocab["<pad>"])
PAD_TOKEN = "<pad>"
PAD_ID    = vocab[PAD_TOKEN]
CLS_ID    = vocab["<cls>"]
PAD_VALUE = model_configs.get("pad_value", -2)
print(f"vocab size {len(vocab)} | pad_id {PAD_ID} | cls_id {CLS_ID} | pad_value {PAD_VALUE}")

glia.var["gene_symbol"] = glia.var_names.astype(str)
glia.var["scgpt_id"] = [vocab[g] if g in vocab else -1 for g in glia.var["gene_symbol"]]
in_vocab = (glia.var["scgpt_id"] >= 0).values
frac_in_vocab = float(in_vocab.mean())
print(f"panel genes in scGPT vocabulary: {int(in_vocab.sum())} / {glia.n_vars} ({frac_in_vocab:.1%})")

# Pre-registered hard gate: without APOE the downstream APOE-axis eval is not runnable.
NICHE_GENES = ["APOE", "TREM2", "GFAP", "AQP4", "CSF1R", "AIF1", "CLU", "MS4A6A"]
panel_genes = set(glia.var["gene_symbol"])
vocab_genes = set(glia.var.loc[in_vocab, "gene_symbol"])
niche_status = {g: {"in_panel": g in panel_genes, "in_vocab": g in vocab_genes} for g in NICHE_GENES}
niche_warnings = [g for g, st in niche_status.items() if not st["in_vocab"]]
print("\nniche-gene survival through the vocabulary:")
for g, st in niche_status.items():
    print(f"  {g:8} in_panel={st['in_panel']} in_vocab={st['in_vocab']}")
assert niche_status["APOE"]["in_vocab"], "APOE is not in the scGPT vocabulary -- the APOE-axis eval is impossible for this FM"
apoe_hard_fail_gate = "passed"

# Cross-check against colab_10's recorded coverage -- recomputed, not read back.
AUDIT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_PATH) as f:
    audit_prior = json.load(f)
_zs_frac = audit_prior.get("scgpt_zeroshot", {}).get("vocab_audit", {}).get("frac_in_vocab")
if _zs_frac is not None:
    print(f"\ncolab_10 recorded frac_in_vocab {_zs_frac:.4f} | recomputed here {frac_in_vocab:.4f}")
    assert abs(_zs_frac - frac_in_vocab) < 1e-3, (
        "recomputed vocabulary coverage disagrees with the zero-shot run -- the gene panel changed")
else:
    print("\nWARNING -- colab_10's frac_in_vocab not found in the audit trail; no cross-check performed")

glia_v = glia[:, in_vocab].copy()
GENE_IDS = glia_v.var["scgpt_id"].to_numpy(dtype=np.int64)
assert (GENE_IDS >= 0).all(), "an out-of-vocabulary gene survived into the in-vocab subset"
assert len(GENE_IDS) == glia_v.n_vars, "gene-id array length does not match the sliced matrix"
del glia; gc.collect()
print("in-vocab substrate:", glia_v.shape)

VOCAB_AUDIT = {"vocab_size": int(len(vocab)), "n_genes_panel": int(len(in_vocab)),
               "n_in_vocab": int(in_vocab.sum()), "frac_in_vocab": round(frac_in_vocab, 4),
               "niche_status": niche_status, "niche_warnings": niche_warnings,
               "apoe_hard_fail_gate": apoe_hard_fail_gate}
_ram("after vocab subset")


vocab size 60697 | pad_id 60694 | cls_id 60695 | pad_value -2
panel genes in scGPT vocabulary: 19977 / 26514 (75.3%)

niche-gene survival through the vocabulary:
  APOE     in_panel=True in_vocab=True
  TREM2    in_panel=True in_vocab=True
  GFAP     in_panel=True in_vocab=True
  AQP4     in_panel=True in_vocab=True
  CSF1R    in_panel=True in_vocab=True
  AIF1     in_panel=True in_vocab=True
  CLU      in_panel=True in_vocab=True
  MS4A6A   in_panel=True in_vocab=True

colab_10 recorded frac_in_vocab 0.7535 | recomputed here 0.7535
in-vocab substrate: (142588, 19977)
[RAM] after vocab subset          :   7.1 / 179.4 GB (5%)


> **Interpretation — vocabulary coverage reproduces colab_10's zero-shot number to four decimal places; every AD-relevant marker gene survives (3a).**
>
> scGPT's checkpoint vocabulary holds 60,697 tokens (special-token ids `pad`=60694, `cls`=60695; `pad_value`=-2 is the sentinel *expression value* for padded positions, not a token id). Of the 26,514 substrate genes, 19,977 (75.3%) are present in that vocabulary -- and this fraction, 0.7535, matches colab_10's originally recorded zero-shot figure to four decimal places, confirming the vocabulary intersection logic here reproduces exactly what every earlier scGPT notebook in this project used, not a subtly different gene-matching rule. All eight niche marker genes checked by name -- APOE, TREM2, GFAP, AQP4, CSF1R, AIF1, CLU, MS4A6A -- are both in the gene panel and in scGPT's vocabulary, so none of the genes this project's biological interpretation most depends on (APOE for the carrier axis, TREM2/CSF1R/AIF1 for microglial activation, GFAP/AQP4 for astrocyte reactivity, CLU/MS4A6A as AD-associated markers) is silently dropped by the vocabulary subsetting. The resulting in-vocab substrate is (142,588 cells, 19,977 genes) -- gene columns drop from 26,514 to 19,977, cell count is untouched.


### 3b — Per-study sequence-length geometry: the premise this notebook's prediction rests on

This is the cell the pre-registration in the header depends on, and it runs **before** any GPU work
so the prediction is read against a measured premise rather than an assumed one.

colab_14's candidate explanation for the Geneformer per-study drift spread was median tokenized
sequence length. scGPT's 1200-token ceiling should largely erase that lever, because a cell above the
cap contributes exactly 1200 tokens no matter how many genes it detected. Three quantities decide
whether that is actually true *here*:

- **pre-cap detected in-vocab genes per study** — the analogue of Geneformer's median tokenized
  length, and the quantity colab_14's spread rank-ordered with;
- **fraction of each study's cells above the cap** — how much of that variation the ceiling absorbs;
- **post-cap effective length per study** (`min(nnz + 1, 1200)`) — what the model actually sees.

The reading rule, fixed now: if post-cap effective lengths are close across studies while pre-cap
gene counts are not, the lever is genuinely flattened and §6b's drift spread is an informative test
of the length explanation. If a study sits mostly *under* the cap, its effective length still varies
and the test is correspondingly weaker for that study — which must be stated, not quietly ignored.

The batch-geometry check is the same arithmetic that cost this project two sessions once: an
attention score tensor is `batch x heads x length^2` elements and a 32-bit element count is a real
ceiling. It is cheap and it fails before the run rather than during it.


In [7]:

MAX_LENGTH = 1200        # whole-human context; the value colab_10/16/17/18 all used
EMB_BATCH  = 64

Xv = glia_v.X
assert sp.issparse(Xv), "expected a sparse in-vocab matrix; densifying the full object is not intended here"
Xv = Xv.tocsr()
Xv.eliminate_zeros()
nnz_per_cell = np.diff(Xv.indptr)
seq_len      = nnz_per_cell + 1                        # +1 for the prepended <cls> token
eff_len      = np.minimum(seq_len, MAX_LENGTH)         # what the model actually sees
over         = int((seq_len > MAX_LENGTH).sum())

n_zero_genes = int((nnz_per_cell == 0).sum())
assert n_zero_genes == 0, (
    f"{n_zero_genes} cells have no detected in-vocab gene -- they cannot be embedded")

print(f"detected in-vocab genes per cell (n={len(nnz_per_cell)}):")
for q in (0.05, 0.25, 0.50, 0.75, 0.95, 1.00):
    print(f"  q{q:<5.2f}: {np.quantile(nnz_per_cell, q):8.0f}")
print(f"\ncells above the {MAX_LENGTH}-token context: {over} ({over/len(seq_len):.1%})")
print("  -> these are RANDOMLY SUBSAMPLED to the context length on every pass unless the")
print("     deterministic-binning path in 5a is in force; either way the SET of genes kept is")
print("     redrawn per pass, which is why detector #1's floor is measured in 5b, not assumed.")

# --- the per-study table the prediction is read against ----------------------------------------
study_v = glia_v.obs["study_id"].astype(str).values
GEOMETRY = {}
print(f"\nper-study input geometry (cap = {MAX_LENGTH} tokens):")
print(f"  {'study':12} {'n_cells':>8} {'pre-cap med':>12} {'over-cap':>9} {'post-cap med':>13} {'post-cap IQR':>16}")
for s in STUDIES:
    m = study_v == s
    pre_med  = float(np.median(nnz_per_cell[m]))
    post_med = float(np.median(eff_len[m]))
    q25, q75 = float(np.quantile(eff_len[m], 0.25)), float(np.quantile(eff_len[m], 0.75))
    frac_over = float((seq_len[m] > MAX_LENGTH).mean())
    GEOMETRY[s] = {"n_cells": int(m.sum()), "pre_cap_median_genes": pre_med,
                   "frac_over_cap": round(frac_over, 4), "post_cap_median_len": post_med,
                   "post_cap_iqr": [q25, q75]}
    print(f"  {s:12} {int(m.sum()):8d} {pre_med:12.0f} {frac_over:8.1%} {post_med:13.0f} "
          f"{f'{q25:.0f}-{q75:.0f}':>16}")

for s in STUDIES:
    if GEOMETRY[s]["frac_over_cap"] < 0.5:
        print(f"  NOTE: {s} has only {GEOMETRY[s]['frac_over_cap']:.1%} of cells over the "
              f"{MAX_LENGTH}-token cap -- effective length still varies meaningfully within this "
              "study, weakening the test specifically for it even if the aggregate premise holds.")

_pre  = np.array([GEOMETRY[s]["pre_cap_median_genes"] for s in STUDIES])
_post = np.array([GEOMETRY[s]["post_cap_median_len"] for s in STUDIES])
PRE_CAP_SPREAD  = float(_pre.max() / _pre.min())
POST_CAP_SPREAD = float(_post.max() / _post.min())
FLATTENING = PRE_CAP_SPREAD / POST_CAP_SPREAD if POST_CAP_SPREAD > 0 else float("inf")
print(f"\npre-cap median spread across studies : {PRE_CAP_SPREAD:.2f}x")
print(f"post-cap median spread across studies: {POST_CAP_SPREAD:.2f}x")
print(f"flattening factor (pre/post, informational only -- see the gate below): {FLATTENING:.2f}x")

# Geneformer's own numbers, for the comparison 6b makes. Hardcoded from colab_14's printed table
# (median tokenized length / population-matched drift_all), not recomputed here.
GF_REFERENCE = {
    "median_len":  {"SEA-AD": 2852, "Li2025": 1612, "Haney2024": 2077},
    "drift_all":   {"SEA-AD": 0.00543, "Li2025": 0.00300, "Haney2024": 0.00359},
    "drift_spread": 1.81,
}
print("\ncolab_14 (Geneformer) for reference -- median tokenized length vs population-matched drift_all:")
for s in STUDIES:
    print(f"  {s:12} median_len {GF_REFERENCE['median_len'][s]:6d} | drift_all {GF_REFERENCE['drift_all'][s]:.5f}")
print(f"  drift_all spread across the three Geneformer checkpoints: {GF_REFERENCE['drift_spread']:.2f}x")

# Gate re-specified pre-run: FLATTENING = PRE/POST is bounded above by
# PRE_CAP_SPREAD since POST_CAP_SPREAD >= 1 always, so a ratio-based threshold can never be reached
# once every study's median already sits at or above the cap -- exactly the case the cap is supposed
# to produce. The premise is read directly off the two quantities instead, matching the reading rule
# stated above: post-cap lengths must sit close to each other AND a real pre-cap difference must have
# existed for "flattened" to mean anything.
MAX_POST_CAP_SPREAD = 1.05   # post-cap medians within 5% of each other across studies
MIN_PRE_CAP_SPREAD  = 1.3    # a real pre-cap difference must exist for the cap to be doing work
PREMISE_OK = bool(POST_CAP_SPREAD <= MAX_POST_CAP_SPREAD and PRE_CAP_SPREAD >= MIN_PRE_CAP_SPREAD)
if PREMISE_OK:
    print(f"\nPREMISE HOLDS: post-cap medians sit within {(POST_CAP_SPREAD-1)*100:.1f}% of each other "
          f"({POST_CAP_SPREAD:.2f}x <= {MAX_POST_CAP_SPREAD}x) despite a real {PRE_CAP_SPREAD:.2f}x "
          f"pre-cap spread (>= {MIN_PRE_CAP_SPREAD}x) -- the context cap has flattened the length "
          "lever, so 6b's drift spread is an informative test of colab_14's length explanation.")
else:
    print(f"\nPREMISE WEAK: post-cap spread {POST_CAP_SPREAD:.2f}x (need <= {MAX_POST_CAP_SPREAD}x) "
          f"and/or pre-cap spread {PRE_CAP_SPREAD:.2f}x (need >= {MIN_PRE_CAP_SPREAD}x) did not both "
          "clear the bar -- 6b's drift spread would not cleanly discriminate the length explanation "
          "from a study-specific one; 6b must report this rather than read the prediction as resolved.")

n_heads = model_configs["nheads"]
elems = EMB_BATCH * n_heads * MAX_LENGTH ** 2
assert elems < 2**31, f"attention tensor {elems:,} elements exceeds int32 -- lower EMB_BATCH"
print(f"\nbatch geometry OK: {EMB_BATCH} x {n_heads} heads x {MAX_LENGTH}^2 = {elems:,} < {2**31:,}")


detected in-vocab genes per cell (n=142588):
  q0.05 :      794
  q0.25 :     1613
  q0.50 :     2418
  q0.75 :     3342
  q0.95 :     4719
  q1.00 :     9228

cells above the 1200-token context: 123112 (86.3%)
  -> these are RANDOMLY SUBSAMPLED to the context length on every pass unless the
     deterministic-binning path in 5a is in force; either way the SET of genes kept is
     redrawn per pass, which is why detector #1's floor is measured in 5b, not assumed.

per-study input geometry (cap = 1200 tokens):
  study         n_cells  pre-cap med  over-cap  post-cap med     post-cap IQR
  SEA-AD          75634         2977    98.0%          1200        1200-1200
  Li2025          46870         1670    69.1%          1200        1086-1200
  Haney2024       20084         2165    82.7%          1200        1200-1200

pre-cap median spread across studies : 1.78x
post-cap median spread across studies: 1.00x
flattening factor (pre/post, informational only -- see the gate below): 1.78x

colab_

> **Interpretation — the pre-registered premise holds on real data: the 1200-token context cap collapses a real 1.78x pre-cap length spread down to 1.00x, clearing the notebook's own gate before any GPU work runs (3b).**
>
> Detected in-vocab genes per cell range widely (q0.05 = 794, median 2,418, q0.95 = 4,719, max 9,228) -- and since scGPT's context cap is 1,200 tokens, 123,112 of 142,588 cells (86.3%) exceed it and have their gene set randomly subsampled down to the cap on every embedding pass. That subsampling is the reason detector #1's noise floor has to be *measured* (5b) rather than assumed to be zero.
>
> The per-study breakdown is the cell this notebook's whole pre-registered test rests on. Before the cap, median sequence length differs sharply by study -- SEA-AD 2,977, Haney2024 2,165, Li2025 1,670 -- a 1.78x spread (2,977/1,670). After the cap, every study's post-cap median collapses to exactly 1,200 (Li2025's IQR alone stretches down to 1,086, the others sit at 1,200-1,200), a 1.00x spread. That collapse is the mechanism colab_14 (Geneformer) never had available: Geneformer's own 4,096-token ceiling sits far above all three of its per-study medians, so it never bound and never flattened anything. Those medians (SEA-AD 2,852, Haney2024 2,077, Li2025 1,612 -- a 1.77x length spread, essentially the same as the 1.78x pre-cap spread measured here) therefore reached the model as they were, and colab_14's per-study drift spread came out at 1.81x. (The 1.81x is the *drift* spread, not the length spread, which is 1.77x -- colab_14's argument rested on the two quantities *ranking* the three studies identically, not on the two summary ratios happening to land near each other.) If scGPT's cap genuinely flattens the length lever, colab_14's length-based explanation predicts scGPT's own per-study drift spread (measured in 6b) should collapse toward 1.00x as well -- and unlike an assumed premise, this cell measures the flattening directly on this run's own real data before any adapter is trained, gating the cell 6b will read: "PREMISE HOLDS" fires because the post-cap spread (1.00x) sits at or below the 1.05x tolerance despite a genuinely large pre-cap spread (1.78x, above the 1.3x threshold needed to call the flattening non-trivial) -- so 6b's result is licensed to speak to colab_14's length hypothesis rather than being reported as an unresolved test.
>
> The batch-geometry guard against the int32 kernel-launch ceiling (`EMB_BATCH x N_HEADS x MAX_LENGTH^2`, per this project's standing GPU-crash-geometry check) computes 64 x 8 x 1,200^2 = 737,280,000 -- comfortably under the 2,147,483,648 limit, so no crash risk from attention-tensor size at this batch size.

## 4 — Per-study continued pretraining



### 4a — Model factory, checkpoint verification, LoRA attachment

Three adapters are trained, so everything that builds a model has to be a function rather than a
one-off block: each study gets a **freshly constructed model, freshly loaded from the same frozen
checkpoint, with a freshly initialised LoRA adapter**. That is what "parallel and independent" means
in the locked design — sequential continual training is a different regime and is not what this is.

Two silent-failure surfaces are asserted rather than assumed, both carried from colab_16:

- **Checkpoint loading is non-strict.** Any parameter whose name or shape does not match is dropped
  without a word, which is exactly how a run ends up continuing to pretrain against a randomly
  initialised expression decoder: loss falls, drift is real, and the whole thing is meaningless.
  Every parameter on the input→loss path must be shown to have come from the checkpoint.
- **LoRA targets are a regex, not a name list.** Without flash-attn, Q/K/V are a single fused
  `in_proj_weight` Parameter, and peft matches list-form `target_modules` by *name suffix* — a plain
  `["self_attn", "linear1", "linear2"]` would silently also catch `ContinuousValueEncoder.linear1`
  and `linear2`, contaminating the adapter with the value encoder. The regex scopes it to the
  transformer encoder layers, and a regex that matched nothing fails silently too, so attachment is
  asserted positively *and* the value encoder is asserted clean.


In [8]:

import re
from scgpt.model import TransformerModel
from peft import LoraConfig, get_peft_model

DEVICE = torch.device("cuda")
SEED   = 0

# PyTorch's attention modules carry an inference fast path that reads `self_attn.in_proj_weight` as
# a raw tensor instead of calling the module's forward. Every forward here runs under autocast and
# both MultiheadAttention.forward and TransformerEncoderLayer.forward already skip that path in
# that case -- disabled explicitly anyway so nothing depends on it implicitly.
torch.backends.mha.set_fastpath_enabled(False)
print("attention fast path enabled:", torch.backends.mha.get_fastpath_enabled())

def build_scgpt(model_configs, vocab):
    """Construct the model with the same arguments the library's embedding path uses."""
    return TransformerModel(
        ntoken=len(vocab), d_model=model_configs["embsize"], nhead=model_configs["nheads"],
        d_hid=model_configs["d_hid"], nlayers=model_configs["nlayers"],
        nlayers_cls=model_configs["n_layers_cls"], n_cls=1, vocab=vocab,
        dropout=model_configs["dropout"], pad_token=model_configs["pad_token"],
        pad_value=model_configs["pad_value"], do_mvc=True, do_dab=False,
        use_batch_labels=False, domain_spec_batchnorm=False, explicit_zero_prob=False,
        use_fast_transformer=False,      # no flash-attn -> PyTorch attention, as in every prior pass
        fast_transformer_backend="flash", pre_norm=False,
    )

RENAME_RULES = {                          # the released checkpoint uses flash-attention naming
    r"self_attn\._impl\.Wqkv\.":     "self_attn.in_proj_",
    r"self_attn\.Wqkv\.":            "self_attn.in_proj_",
    r"self_attn\._impl\.out_proj\.": "self_attn.out_proj.",
}
REQUIRED_PREFIXES = ("encoder.", "value_encoder.", "transformer_encoder.", "decoder.")

def load_checkpoint_verbose(model, ckpt_path, quiet=False):
    """Load the checkpoint and REPORT what was dropped -- the loader itself drops silently."""
    raw = torch.load(ckpt_path, map_location="cpu")
    renamed = {}
    for k, v in raw.items():
        kk = k
        for pat, rep in RENAME_RULES.items():
            kk = re.sub(pat, rep, kk)
        renamed[kk] = v
    model_dict = model.state_dict()
    kept    = {k: v for k, v in renamed.items() if k in model_dict and v.shape == model_dict[k].shape}
    dropped = sorted(set(renamed) - set(kept))
    model_dict.update(kept)
    model.load_state_dict(model_dict)
    if not quiet:
        print(f"  checkpoint parameters loaded: {len(kept)} | dropped: {len(dropped)}"
              + (f" ({', '.join(dropped[:3])}{' ...' if len(dropped) > 3 else ''})" if dropped else ""))
    return kept, dropped

def assert_required_loaded(model, kept, tag):
    """Every parameter on the input->loss path must have come from the checkpoint."""
    missing = [n for n, _ in model.named_parameters()
               if n.startswith(REQUIRED_PREFIXES) and n not in kept]
    assert not missing, (
        f"[{tag}] {len(missing)} parameter(s) on the input->loss path were NOT loaded from the "
        f"checkpoint, e.g. {missing[:8]} -- CPT against randomly initialised weights is meaningless")
    print(f"  [{tag}] all required parameter groups loaded from the checkpoint")

# ---------------------------------------------------------------- LoRA, identical to colab_16
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
LORA_TARGETS_REGEX = r"transformer_encoder\.layers\.\d+\.(self_attn|linear1|linear2)$"

D, D_HID, N_LAYERS = model_configs["embsize"], model_configs["d_hid"], model_configs["nlayers"]
#   fused input projection r*(d) + r*(3d) = 4rd | out_proj 2rd | linear1 + linear2 = 2r(d + d_hid)
EXPECTED_TRAINABLE = N_LAYERS * (4*LORA_R*D + 2*LORA_R*D + 2*LORA_R*(D + D_HID))

def build_cpt_model(tag, seed=SEED):
    """A fresh base + a fresh LoRA adapter. Seeded BEFORE get_peft_model, because LoRA's A matrices
    are Kaiming-initialised there -- seeding afterwards would overstate reproducibility."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = build_scgpt(model_configs, vocab)
    kept, _ = load_checkpoint_verbose(model, os.path.join(MODEL_DIR, "best_model.pt"))
    assert_required_loaded(model, kept, tag)
    pm = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA,
                                          lora_dropout=LORA_DROPOUT,
                                          target_modules=LORA_TARGETS_REGEX, bias="none"))
    l0 = model.transformer_encoder.layers[0]
    for name, mod in (("self_attn", l0.self_attn), ("linear1", l0.linear1), ("linear2", l0.linear2)):
        assert hasattr(mod, "lora_A"), f"[{tag}] LoRA did not attach to {name} -- target regex matched nothing"
    leak = [n for n, m in model.named_modules() if n.startswith("value_encoder") and hasattr(m, "lora_A")]
    assert not leak, f"[{tag}] LoRA leaked into value_encoder despite the scoped regex: {leak}"
    actual = sum(p.numel() for p in pm.parameters() if p.requires_grad)
    assert actual == EXPECTED_TRAINABLE, (
        f"[{tag}] trainable params {actual:,} != derived {EXPECTED_TRAINABLE:,} -- the adapter did "
        "not attach where this notebook's accounting assumes it did")
    print(f"  [{tag}] LoRA attached, trainable {actual:,} (matches derivation)")
    return pm, model

_probe_pm, _probe_raw = build_cpt_model("attachment probe")
n_frozen = sum(p.numel() for p in _probe_pm.parameters() if not p.requires_grad)
print(f"\nfrozen base parameters: {n_frozen:,} | adapted fraction: "
      f"{EXPECTED_TRAINABLE/(n_frozen+EXPECTED_TRAINABLE):.4%}")
del _probe_pm, _probe_raw; gc.collect(); torch.cuda.empty_cache()


attention fast path enabled: False
  checkpoint parameters loaded: 162 | dropped: 1 (flag_encoder.weight)
  [attachment probe] all required parameter groups loaded from the checkpoint
  [attachment probe] LoRA attached, trainable 491,520 (matches derivation)

frozen base parameters: 51,856,898 | adapted fraction: 0.9389%


> **Interpretation — the model factory attaches LoRA correctly on a first probe pass: same 491,520 trainable parameters, same one dropped checkpoint tensor, as colab_16's aggregated run (4a).**
>
> With the attention fast path disabled (PyTorch's fused-attention inference path reads `self_attn.in_proj_weight` as a raw tensor instead of calling the module's forward, which is precisely how a LoRA-wrapped attention module can end up bypassed and contributing nothing without raising; whether the target regex *matches* is unaffected either way), the checkpoint loads 162 of its parameter groups and drops exactly one: `flag_encoder.weight`, the same single tensor every scGPT notebook in this project has dropped -- it belongs to a training-time auxiliary head this project's fine-tuning recipe does not use. LoRA then attaches 491,520 trainable parameters against a frozen base of 51,856,898 (0.9389% adapted) -- identical to colab_16's aggregated-regime figures, confirming the per-study factory function builds a structurally identical model to the one already validated there; only the training data slice differs between the three calls this factory will make in 4c, not the model architecture or adapter configuration.


### 4b — Dataset, collator, the epoch-matched budget, and `run_study()`

**Training-budget rule (per-study epoch matching), inherited from colab_14 so the two per-study
regimes are directly comparable.** colab_16 trained 2000 steps at effective batch 32 over 94,963
train cells = 0.674 epochs. Each study here gets the step count that reproduces that same per-cell
exposure on its own slice:

```
steps_S = round(TARGET_EPOCHS x n_train_S / effective_batch),   TARGET_EPOCHS = 0.674
```

Two consequences, both intended: every study is trained to the same intensity per cell, with
gradients coming from one study at a time rather than a shuffled mixture; and because the three
slices partition the train split, the steps **sum to ca. 2000 across the three adapters together** —
the same total gradient budget colab_16 spent on one.

**What epoch matching does not fix, restated because it is easy to lose.** Each individual adapter is
*not* budget-matched to colab_16's aggregated checkpoint it gets compared against in §6b: SEA-AD gets
ca. 54% of colab_16's update count, Li2025 ca. 33%, Haney2024 ca. 13%. Epoch matching removes the
per-cell-exposure confound, not the per-adapter optimisation-budget confound. With a linear decay
schedule the shorter runs also traverse the whole schedule, so they are not simply truncated
versions of the longer ones.

Training runs under the library's **stock** binning. That is deliberate: colab_16's aggregated
adapter was fitted under stock binning, so training these three the same way keeps the regime
contrast clean. The deterministic-binning patch is installed in §5a, *after* all training is done and
before any embedding pass — so measurement is on one common path while training parity is preserved.


In [9]:

import math, time
from torch.utils.data import DataLoader, SequentialSampler, RandomSampler
from scgpt.data_collator import DataCollator
from scgpt.loss import masked_mse_loss
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

import scgpt.preprocess as _sp_guard
_STOCK_DIGITIZE = _sp_guard._digitize   # captured now, before 5a can possibly patch it

# ---------------------------------------------------------------- CPT configuration (colab_16 parity)
MASK_VALUE     = -1
MLM_PROB       = 0.15
LEARNING_RATE  = 5e-4
PER_DEV_BATCH  = 8
GRAD_ACCUM     = 4
EFF_BATCH      = PER_DEV_BATCH * GRAD_ACCUM      # 32
WARMUP_RATIO   = 0.05
NUM_WORKERS    = 2
VAL_EVAL_CAP   = 2000
VAL_EVAL_SEED  = 0

assert MASK_VALUE != PAD_VALUE, (
    f"mask sentinel {MASK_VALUE} equals the pad value {PAD_VALUE}; masked positions would include "
    "padding and the loss would be computed over it")

# epoch target inherited from the aggregated run so per-cell exposure matches across regimes AND
# across the two FMs (colab_14 used the same figure for Geneformer).
AGG_STEPS, AGG_TRAIN_CELLS = 2000, REF_N_CELLS_SP["train"]
TARGET_EPOCHS = AGG_STEPS * EFF_BATCH / AGG_TRAIN_CELLS
print(f"TARGET_EPOCHS inherited from the aggregated run: {TARGET_EPOCHS:.4f}")

class GliaCellDataset(torch.utils.data.Dataset):
    """One cell -> (gene ids, expression values) with <cls> prepended, densified per row."""
    def __init__(self, X_csr, gene_ids, cls_id, pad_value):
        self.X, self.gene_ids, self.cls_id, self.pad_value = X_csr, gene_ids, cls_id, pad_value
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        row = self.X[idx].toarray().ravel()
        nz = np.nonzero(row)[0]
        genes  = np.insert(self.gene_ids[nz], 0, self.cls_id)
        values = np.insert(row[nz], 0, self.pad_value)
        return {"id": idx,
                "genes": torch.from_numpy(genes).long(),
                "expressions": torch.from_numpy(values).float()}

full_ds = GliaCellDataset(Xv, GENE_IDS, CLS_ID, PAD_VALUE)

def make_collator(do_mlm):
    return DataCollator(
        do_padding=True, pad_token_id=PAD_ID, pad_value=PAD_VALUE,
        do_mlm=do_mlm, do_binning=True, mlm_probability=MLM_PROB,
        mask_value=MASK_VALUE, max_length=MAX_LENGTH,
        sampling=True, keep_first_n_tokens=1,
    )

split_vals = glia_v.obs["split"].astype(str).values
study_vals = glia_v.obs["study_id"].astype(str).values
lineage_vals = glia_v.obs["lineage"].astype(str).values

# --- per-study step budget, derived and printed before anything trains --------------------------
STEP_BUDGET = {}
print(f"\n{'study':12} {'n_train':>8} {'steps':>6} {'epochs':>8} {'% of colab_16 steps':>20}")
for s in STUDIES:
    n_tr = int(((split_vals == "train") & (study_vals == s)).sum())
    steps = max(1, int(round(TARGET_EPOCHS * n_tr / EFF_BATCH)))
    if SMOKE:
        steps = 8
    STEP_BUDGET[s] = {"n_train": n_tr, "max_steps": steps,
                      "epochs": steps * EFF_BATCH / max(n_tr, 1)}
    print(f"{s:12} {n_tr:8d} {steps:6d} {STEP_BUDGET[s]['epochs']:8.3f} {steps/AGG_STEPS:19.1%}")
_total_steps = sum(v["max_steps"] for v in STEP_BUDGET.values())
print(f"\ntotal steps across the three adapters: {_total_steps} "
      f"(colab_16 aggregated spent {AGG_STEPS} on one)")
if not SMOKE:
    assert abs(_total_steps - AGG_STEPS) <= 3, (
        f"per-study steps sum to {_total_steps}, not ~{AGG_STEPS} -- the slices should partition the "
        "train split, so the summed-budget equivalence should hold to rounding")

USE_BF16  = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("autocast dtype:", AMP_DTYPE)

def run_study(study, log_every=50, eval_every=250):
    """Train one independent adapter on one study's train slice. Returns a result record."""
    assert _sp_guard._digitize is _STOCK_DIGITIZE, (
        "scgpt.preprocess._digitize has already been patched (5a's deterministic-binning fix) -- "
        "training must run under stock binning to stay comparable to colab_16's aggregated adapter "
        "fit; re-run this cell before 5a, not after")
    slug   = SLUG[study]
    budget = STEP_BUDGET[study]
    max_steps = budget["max_steps"]
    le = 2 if SMOKE else log_every
    ee = 4 if SMOKE else max(1, min(eval_every, max_steps // 8))
    print(f"\n{'='*78}\n{study} -- {budget['n_train']} train cells, {max_steps} steps, "
          f"{budget['epochs']:.3f} epochs\n{'='*78}")

    pm, raw = build_cpt_model(study)
    pm.to(DEVICE)

    tr_idx = np.flatnonzero((split_vals == "train") & (study_vals == study))
    va_idx = np.flatnonzero((split_vals == "val")   & (study_vals == study))
    assert len(tr_idx) > 0, f"{study}: empty train slice"
    assert len(va_idx) > 0, (
        f"{study}: no val cells -- the frozen split gives this study no validation slice, so the "
        "monitoring curve for it cannot be built and undertraining would not be diagnosable")

    # capped, lineage-stratified val subsample: monitoring only, never a substitute for the full
    # substrate that detector #1 embeds in section 5.
    rng_v = np.random.default_rng(VAL_EVAL_SEED)
    lin_va = lineage_vals[va_idx]
    parts = []
    for lin in sorted(np.unique(lin_va)):
        li = va_idx[lin_va == lin]
        take = min(int(round(VAL_EVAL_CAP * len(li) / len(va_idx))), len(li))
        if take > 0:
            parts.append(rng_v.choice(li, size=take, replace=False))
    va_eval_idx = np.sort(np.concatenate(parts)) if parts else va_idx[:1]
    print(f"  train {len(tr_idx)} | val {len(va_idx)} (monitoring subsample {len(va_eval_idx)})")

    torch.manual_seed(SEED); np.random.seed(SEED)
    # one Subset object per loader -- the sampler must index the SAME object the loader reads,
    # not a second Subset that merely happens to have the same length.
    tr_ds = torch.utils.data.Subset(full_ds, tr_idx)
    va_ds = torch.utils.data.Subset(full_ds, va_eval_idx)
    train_loader = DataLoader(tr_ds, batch_size=PER_DEV_BATCH, sampler=RandomSampler(tr_ds),
                              collate_fn=make_collator(True), drop_last=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    # num_workers=0 + a reseed before each pass -> the validation mask draw is identical every time.
    val_loader   = DataLoader(va_ds, batch_size=PER_DEV_BATCH, sampler=SequentialSampler(va_ds),
                              collate_fn=make_collator(True), drop_last=False,
                              num_workers=0, pin_memory=True)

    optimizer = AdamW([p for p in pm.parameters() if p.requires_grad], lr=LEARNING_RATE)
    warmup = max(1, int(WARMUP_RATIO * max_steps))
    scheduler = LambdaLR(optimizer, lambda st: st / max(1, warmup) if st < warmup
                         else max(0.0, (max_steps - st) / max(1, max_steps - warmup)))

    def forward_loss(batch):
        gene   = batch["gene"].to(DEVICE, non_blocking=True)
        target = batch["expr"].to(DEVICE, non_blocking=True)
        masked = batch["masked_expr"].to(DEVICE, non_blocking=True)
        pad_mask = gene.eq(PAD_ID)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            out = raw(src=gene, values=masked, src_key_padding_mask=pad_mask,
                      CLS=False, CCE=False, MVC=False, ECS=False)
        positions = masked.eq(MASK_VALUE)
        if positions.sum() == 0:
            return None, 0
        return masked_mse_loss(out["mlm_output"].float(), target.float(), positions), int(positions.sum())

    @torch.no_grad()
    def evaluate():
        raw.eval()
        torch.manual_seed(1234)        # fixed mask draw -> the val curve is comparable across steps
        tot, n = 0.0, 0
        for b in val_loader:
            loss, npos = forward_loss(b)
            if loss is None:
                continue
            tot += float(loss.detach()) * npos; n += npos
        raw.train()
        return tot / max(n, 1)

    history, raw_train = [], []
    raw.train()
    step, running, running_n, t0 = 0, 0.0, 0, time.time()
    it = iter(train_loader)
    while step < max_steps:
        optimizer.zero_grad(set_to_none=True)
        acc = 0.0
        for _ in range(GRAD_ACCUM):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(train_loader); batch = next(it)
            loss, npos = forward_loss(batch)
            if loss is None:
                continue
            (loss / GRAD_ACCUM).backward()
            acc += float(loss.detach()) / GRAD_ACCUM
        torch.nn.utils.clip_grad_norm_([p for p in pm.parameters() if p.requires_grad], 1.0)
        optimizer.step(); scheduler.step(); step += 1
        running += acc; running_n += 1; raw_train.append(acc)
        if step % le == 0:
            rec = {"step": step, "train_loss": running / running_n, "lr": scheduler.get_last_lr()[0]}
            history.append(rec)
            print(f"  step {step:5d} | train {rec['train_loss']:.4f} | lr {rec['lr']:.2e} | {time.time()-t0:.0f}s")
            running, running_n = 0.0, 0
        if step % ee == 0 or step == max_steps:
            vl = evaluate()
            history.append({"step": step, "eval_loss": vl})
            print(f"  step {step:5d} | VAL masked-MSE {vl:.4f}")

    secs = time.time() - t0
    evals = [(r["step"], r["eval_loss"]) for r in history if "eval_loss" in r]
    logged = [r["train_loss"] for r in history if "train_loss" in r]
    adapter_dir = os.path.join(DRIVE_ROOT, "scgpt", f"cpt_per_study_{slug}_{RUN_TAG}_adapter{SUFFIX}")
    os.makedirs(adapter_dir, exist_ok=True)
    pm.save_pretrained(adapter_dir)
    print(f"  finished in {secs/60:.1f} min | train mean {np.mean(raw_train):.4f} "
          f"final-window {logged[-1] if logged else float('nan'):.4f} | "
          f"val first {evals[0][1]:.4f} -> final {evals[-1][1]:.4f}")
    print(f"  saved adapter -> {adapter_dir}")

    del pm, raw, optimizer, scheduler, train_loader, val_loader
    gc.collect(); torch.cuda.empty_cache()

    return {"study": study, "slug": slug,
            "n_train": budget["n_train"], "n_val": int(len(va_idx)),
            "max_steps": max_steps, "epochs": round(budget["epochs"], 4),
            "train_seconds": round(secs, 1),
            "train_loss_mean": float(np.mean(raw_train)),
            "train_loss_final_window": float(logged[-1]) if logged else None,
            "eval_loss_first": float(evals[0][1]), "eval_loss_final": float(evals[-1][1]),
            "eval_curve": [(int(s), round(float(v), 4)) for s, v in evals],
            "adapter_file": os.path.basename(adapter_dir)}


TARGET_EPOCHS inherited from the aggregated run: 0.6739

study         n_train  steps   epochs  % of colab_16 steps
SEA-AD          51218   1079    0.674               53.9%
Li2025          31349    660    0.674               33.0%
Haney2024       12396    261    0.674               13.1%

total steps across the three adapters: 2000 (colab_16 aggregated spent 2000 on one)
autocast dtype: torch.bfloat16


> **Interpretation — the epoch-matched step budget derives cleanly from each study's own slice size and sums exactly to colab_16's 2000 steps (4b).**
>
> `TARGET_EPOCHS = 0.6739` is inherited from colab_16's aggregated run (2000 steps at effective batch 32 over 94,963 cells = 0.674 epochs), and each study's step count is `round(0.6739 x n_train_S / 32)`: SEA-AD (51,218 train cells) gets 1,079 steps, Li2025 (31,349) gets 660, Haney2024 (12,396) gets 261. Each study lands at the same 0.674-epoch exposure per cell -- meaning every training cell in every study is seen roughly the same fractional number of times, which is the sense in which the three runs are comparable despite training on very different slice sizes. The three step counts sum to exactly 2,000 -- the same total steps colab_16 spent on one combined adapter, now split three ways in proportion to slice size rather than spent all at once on the pooled, shuffled data. One direct consequence of this rule, worth carrying into 6b's confound discussion: because steps scale with `n_train`, SEA-AD trains for roughly 4.1x as many steps as Haney2024 (1,079 vs 261) -- so "epoch-matched" equalizes per-cell exposure, not optimisation budget, and the three checkpoints differ in absolute training time as well as in which cells they saw. Training runs under `bfloat16` autocast, the same numeric precision as every other scGPT training pass in this project.


### 4c — Train the three adapters

Sequential in wall-clock, independent in gradients: each call rebuilds the model from the frozen
checkpoint, so no adapter inherits anything from the one before it. The validation curve is captured
per study for the same reason colab_16 captured it — so an undertraining confound is *diagnosable*
rather than assumed, and so the thinnest study (Haney2024, ca. 261 steps) can be checked for having
plateaued rather than merely stopped.


In [10]:

results = {}
_t_all = time.time()
for s in STUDIES:
    results[s] = run_study(s)
    _ram(f"after {s}")
print(f"\n{'='*78}\nall three adapters trained in {(time.time()-_t_all)/60:.1f} min total")

summary = pd.DataFrame([
    {"study": s, "n_train": r["n_train"], "steps": r["max_steps"], "epochs": r["epochs"],
     "min": round(r["train_seconds"]/60, 1), "train_loss_mean": round(r["train_loss_mean"], 4),
     "val_first": round(r["eval_loss_first"], 4), "val_final": round(r["eval_loss_final"], 4),
     "val_delta": round(r["eval_loss_final"] - r["eval_loss_first"], 4)}
    for s, r in results.items()]).set_index("study").loc[STUDIES]
print("\n", summary.to_string())
print("\nval_delta < 0 means the curve was still improving (loss falling) between the first and last")
print("checkpoint. The undertraining signature to check for is a curve STILL FALLING substantially at")
print("the budget's end -- a strongly negative val_delta with no sign of flattening -- especially for")
print("the thinnest run, Haney2024. A val_delta at or near 0 means the curve had already plateaued,")
print("which is the OPPOSITE of undertraining, before reading any per-study difference in section 6")
print("as a regime effect.")



SEA-AD -- 51218 train cells, 1079 steps, 0.674 epochs
  checkpoint parameters loaded: 162 | dropped: 1 (flag_encoder.weight)
  [SEA-AD] all required parameter groups loaded from the checkpoint
  [SEA-AD] LoRA attached, trainable 491,520 (matches derivation)
  train 51218 | val 10911 (monitoring subsample 2000)
  step    50 | train 180.3660 | lr 4.72e-04 | 17s
  step   100 | train 148.0414 | lr 4.77e-04 | 32s
  step   134 | VAL masked-MSE 146.3523
  step   150 | train 148.2145 | lr 4.53e-04 | 58s
  step   200 | train 146.3860 | lr 4.28e-04 | 73s
  step   250 | train 147.2902 | lr 4.04e-04 | 89s
  step   268 | VAL masked-MSE 145.2818
  step   300 | train 146.8301 | lr 3.80e-04 | 114s
  step   350 | train 145.8685 | lr 3.55e-04 | 130s
  step   400 | train 145.2853 | lr 3.31e-04 | 146s
  step   402 | VAL masked-MSE 145.0565
  step   450 | train 146.1253 | lr 3.07e-04 | 172s
  step   500 | train 145.4409 | lr 2.82e-04 | 188s
  step   536 | VAL masked-MSE 144.9189
  step   550 | train 146.1

> **Interpretation — all three adapters plateaued before their budget ran out, including the thinnest run (Haney2024); undertraining ruled out as a confound for section 6 (4c).**
>
> Each study trains independently from the same frozen checkpoint (LoRA re-attached fresh per call, so no adapter inherits gradients from the one trained before it), for its 4b-derived step count: SEA-AD 1,079 steps / 7.1 min, Li2025 660 steps / 4.8 min, Haney2024 261 steps / 2.8 min -- 14.8 min total. Validation masked-MSE is checked periodically within each run for the same reason colab_16 checked it: to make an undertraining confound diagnosable rather than assumed away.
>
> The recorded `val_delta` (final validation minus first, so a negative value means the loss fell over the run) is largest in magnitude for Haney2024 (-4.26) and smallest for SEA-AD (-1.48), which at first glance could read as "the thinnest run was still improving fastest." Reading the printed step-by-step values directly shows the opposite: Haney2024's validation loss falls sharply and early (156.22 at step 32 to 152.37 by step 96, most of the total -4.26 delta), then oscillates in a narrow band for its remaining ca. 165 steps (152.52, 152.35, 152.00, 151.49, 152.03, 151.96) -- noise around a plateau, not a still-falling curve. SEA-AD and Li2025 show the same shape at their own scales: both curves flatten well before their final checkpoint, with SEA-AD's last four validation reads (144.76, 144.70, 144.97, 144.87) spanning under 0.3 units and Li2025's last four (157.11, 157.14, 157.14, 157.18) spanning under 0.1. The signature that would actually indicate undertraining -- a curve still falling substantially at the budget's end -- is absent in all three, so none of the per-study drift differences read in section 6 can be attributed to one adapter simply having had less time to converge than the others.

## 5 — Embedding passes on one common, deterministic path



### 5a — Install the deterministic-binning patch, then verify it actually closed the gap

scGPT's quantile-binning step (`scgpt.preprocess._digitize`) breaks ties between genes landing on the
same bin boundary by drawing an **unseeded** `np.random.rand()` on every call, for every cell — not
only for the over-context cells whose gene *subsampling* is the intended source of pass-to-pass
randomness. colab_18 found this live and replaced it with a fixed midpoint rule.

**Why the patch goes here and not earlier.** Training (§4) ran under the library's stock binning,
matching how colab_16's aggregated adapter was fitted — so the regime contrast in §6b is not
confounded by a change in the training encoding. Every *measurement* from this point on runs through
the patched path instead: the fresh base reference, the noise floor, all three per-study checkpoints
and the re-embedded aggregated adapter. One common encoding for everything being compared.

**What this costs and what it buys.** The absolute numbers below are not directly comparable to the
embeddings colab_16/17 stored, which were produced under the stock random tie-break — so §5b builds
its own base reference on this path rather than treating colab_10's stored zero-shot file as the
reference, and §5c re-embeds colab_16's adapter here rather than reusing its stored numbers. In
exchange, the measured floor stops absorbing a stochasticity source that has nothing to do with the
biology, and the gap between this run's floor and colab_16's 0.00365 is itself the direct answer to
the mechanism-attribution question colab_18 left open — how much of that floor was gene subsampling
and how much was binning.

**Fork ordering matters.** The embedding `DataLoader`s below use `num_workers > 0`, and on Linux
those workers are forked *after* this cell runs, so each inherits the already-patched function. A
patch applied after workers were spawned, or under the `spawn` start method, would silently fail to
reach the processes actually doing the binning.


In [11]:

import scgpt.preprocess as _sp

DETERMINISTIC_BINNING = True     # see 5a; False reproduces colab_16/17's stock encoding instead
_DIGITIZE_STOCK = _sp._digitize

def _digitize_deterministic(x: np.ndarray, bins: np.ndarray, side: str = "both") -> np.ndarray:
    """Deterministic replacement for scgpt.preprocess._digitize (colab_18). The original spreads
    tied quantile-bin assignments with an unseeded np.random.rand(len(x)), redrawn on every call,
    so every cell -- not just over-context ones -- got a different bin assignment on every pass."""
    left_digits = np.digitize(x, bins)
    if side == "one":
        return left_digits
    right_digits = np.digitize(x, bins, right=True)
    digits = 0.5 * (right_digits - left_digits) + left_digits
    return np.ceil(digits).astype(np.int64)

if DETERMINISTIC_BINNING:
    _sp._digitize = _digitize_deterministic
    print("patched scgpt.preprocess._digitize: random tie-break -> deterministic midpoint")
else:
    _sp._digitize = _DIGITIZE_STOCK
    print("WARNING -- running on the STOCK random tie-break; the measured floor in 5b will absorb")
    print("  binning randomness as well as gene subsampling, as colab_16/17's did")

@torch.no_grad()
def embed_cells(model, dataset, batch_size=EMB_BATCH, num_workers=NUM_WORKERS, desc=""):
    """<cls>-position cell embeddings, L2-normalized -- the readout convention every scGPT
    embedding in this project uses."""
    from tqdm.auto import tqdm
    loader = DataLoader(dataset, batch_size=batch_size, sampler=SequentialSampler(dataset),
                        collate_fn=make_collator(False), drop_last=False,
                        num_workers=num_workers, pin_memory=True)
    model.eval()
    out = np.zeros((len(dataset), model_configs["embsize"]), dtype=np.float32)
    count = 0
    for batch in tqdm(loader, desc=desc or "embedding"):
        gene = batch["gene"].to(DEVICE, non_blocking=True)
        expr = batch["expr"].to(DEVICE, non_blocking=True)
        pad_mask = gene.eq(PAD_ID)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            h = model._encode(gene, expr, src_key_padding_mask=pad_mask, batch_labels=None)
        e = h[:, 0, :].float().cpu().numpy()          # <cls> position
        out[count:count + len(e)] = e
        count += len(e)
    assert count == len(dataset), f"embedded {count} of {len(dataset)} cells"
    return out / (np.linalg.norm(out, axis=1, keepdims=True) + 1e-12)

def per_cell_cosine(A, B):
    return (A * B).sum(1) / (np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12)

# --- does the patch actually make an UNDER-CAP cell deterministic? -------------------------------
# Under-cap cells are never gene-subsampled, so with binning fixed two passes over them must agree
# to float32 roundoff (ca. 1e-7). Over-cap cells legitimately still differ -- that is the sampling
# stochasticity 5b is there to measure, and this check must not be run on them or it proves nothing.
# Run at NUM_WORKERS (not 0) so this actually exercises the fork-inheritance path described above --
# at num_workers=0 the patch trivially holds in-process and this check cannot see a fork-order failure.
_under = np.flatnonzero(seq_len <= MAX_LENGTH)
if len(_under) >= 64:
    _probe_idx = _under[:min(512, len(_under))]
    # a PLAIN base model, not a LoRA-wrapped one -- this probes the INPUT ENCODING, so nothing
    # about adapter state or dropout should be able to enter the comparison.
    _probe_model = build_scgpt(model_configs, vocab)
    _pk, _ = load_checkpoint_verbose(_probe_model, os.path.join(MODEL_DIR, "best_model.pt"), quiet=True)
    assert_required_loaded(_probe_model, _pk, "determinism probe")
    _probe_model.to(DEVICE)
    _sub = torch.utils.data.Subset(full_ds, _probe_idx)
    _a = embed_cells(_probe_model, _sub, desc="determinism A")
    _b = embed_cells(_probe_model, _sub, desc="determinism B")
    _maxabs = float(np.max(np.abs(_a - _b)))
    print(f"\ndeterminism check on {len(_probe_idx)} UNDER-cap cells (NUM_WORKERS={NUM_WORKERS}): max |A-B| = {_maxabs:.3e}")
    if DETERMINISTIC_BINNING:
        assert _maxabs < 1e-4, (
            f"under-cap cells still differ by {_maxabs:.3e} between two passes -- the binning patch "
            "did not reach the code actually doing the work (check DataLoader worker fork order)")
        print("  -> under-cap cells are now deterministic; residual is float32 roundoff")
    del _probe_model, _a, _b; gc.collect(); torch.cuda.empty_cache()
else:
    _maxabs = None
    print("\nfewer than 64 under-cap cells available -- determinism check skipped")
DETERMINISM_MAXABS = _maxabs


patched scgpt.preprocess._digitize: random tie-break -> deterministic midpoint
  [determinism probe] all required parameter groups loaded from the checkpoint


determinism A:   0%|          | 0/8 [00:00<?, ?it/s]

determinism B:   0%|          | 0/8 [00:00<?, ?it/s]


determinism check on 512 UNDER-cap cells (NUM_WORKERS=2): max |A-B| = 0.000e+00
  -> under-cap cells are now deterministic; residual is float32 roundoff


> **Interpretation — the deterministic-binning patch verified working exactly as intended: bitwise-identical repeat embeddings on under-cap cells (5a).**
>
> This installs the same fix colab_18 found live: scGPT's `_digitize` breaks ties at bin boundaries with an unseeded `np.random.rand()` on every cell, every call -- not only the over-context cells whose gene *subsampling* is meant to be this notebook's intended source of pass-to-pass randomness. The patch replaces that tie-break with a fixed midpoint rule, applied identically on every call. On a probe of 512 under-cap cells (`NUM_WORKERS=2`, so the patch has to survive worker-process forking, which happens after this cell runs and inherits the already-patched function via fork), two repeat embeddings match at exactly `0.000e+00` -- not merely inside the assertion's 1e-4 tolerance, and not even at the ca. 1e-7 float32 roundoff the cell's own printed note allows for, but bitwise identical. That the same comparison would have been non-zero under colab_16/17's stock binning is the premise this patch rests on rather than something measured here: the stock path was not re-run for this probe, so the counterfactual is an expectation, not a number in this notebook. With it confirmed here, any pass-to-pass variation measured later in this notebook can be attributed to gene subsampling on over-cap cells alone, which is the assumption 5b's floor measurement and 6a's drift readings both depend on.


### 5b — The base reference on this path, and the measured noise floor

Detector #1 asks whether a checkpoint's embedding differs from the unadapted one. For that to have an
answer, the amount two embeddings differ **when nothing has changed** must be known, and for scGPT
that quantity is not zero: cells above the context cap get a redrawn gene subset on every pass.

The reference used from here on is a **freshly embedded frozen base on this notebook's own path**,
not colab_10's stored zero-shot file, because the stored file was produced under the stock binning
this notebook has now patched — using it as the reference would fold an encoding change into every
drift number and make it indistinguishable from adaptation. The stored file is still loaded and
compared, but as a *diagnostic*: the difference between the stored-baseline floor and the repeat-pass
floor is exactly the size of the encoding change, and reporting it is what closes colab_18's open
mechanism-attribution question rather than leaving it as an inference.


In [12]:

# --- stored zero-shot baseline, aligned by cell_index (diagnostic, not the reference) ------------
ZEROSHOT_PATH = os.path.join(DRIVE_ROOT, "scgpt", "glia_scgpt_zeroshot.h5ad")
assert os.path.exists(ZEROSHOT_PATH), f"missing zero-shot baseline {ZEROSHOT_PATH}"
zs = ad.read_h5ad(ZEROSHOT_PATH)
zs_X = zs.X.toarray() if sp.issparse(zs.X) else np.asarray(zs.X)
zs_aligned = pd.DataFrame(np.asarray(zs_X, dtype=np.float32),
                          index=zs.obs["cell_index"].values).reindex(glia_v.obs["cell_index"].values)
assert zs_aligned.notna().all().all(), "baseline rows missing after cell_index alignment"
X_ZS_STORED = zs_aligned.to_numpy(dtype=np.float32)

# cell_index alone can only catch a MISSING cell -- both files use arange(n_obs), so a
# permuted-but-complete index would still pass the notna check. Cross-check the labels that travel
# on both files at every aligned row; this is what would catch a concatenation-order change.
LABEL_COLS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id"]
zs_obs = zs.obs.set_index("cell_index").reindex(glia_v.obs["cell_index"].values)
for col in LABEL_COLS:
    if col not in zs_obs.columns:
        continue
    n_mm = int((zs_obs[col].astype(str).values != glia_v.obs[col].astype(str).values).sum())
    assert n_mm == 0, (
        f"{n_mm} cells have a different '{col}' between the stored baseline and this substrate "
        "after cell_index alignment -- the two files disagree about which row is which cell")
print(f"stored zero-shot labels agree at every aligned row ({', '.join(LABEL_COLS)})")
del zs, zs_X, zs_aligned, zs_obs; gc.collect()

# --- fresh frozen base, twice, on THIS path -----------------------------------------------------
base_model = build_scgpt(model_configs, vocab)
_k, _ = load_checkpoint_verbose(base_model, os.path.join(MODEL_DIR, "best_model.pt"))
assert_required_loaded(base_model, _k, "base reload")
base_model.to(DEVICE)

test_mask = (glia_v.obs["split"] == "test").values
assert test_mask.any(), "no test-split cells -- cannot gate detector #1 on held-out"

X_BASE_A = embed_cells(base_model, full_ds, desc="base pass A")
X_BASE_B = embed_cells(base_model, full_ds, desc="base pass B")
del base_model; gc.collect(); torch.cuda.empty_cache()

floor_repeat_all  = 1.0 - float(np.median(per_cell_cosine(X_BASE_A, X_BASE_B)))
floor_repeat_test = 1.0 - float(np.median(per_cell_cosine(X_BASE_A, X_BASE_B)[test_mask]))
floor_stored_all  = 1.0 - float(np.median(per_cell_cosine(X_ZS_STORED, X_BASE_A)))
floor_stored_test = 1.0 - float(np.median(per_cell_cosine(X_ZS_STORED, X_BASE_A)[test_mask]))

NOISE_FLOOR = floor_repeat_test        # same path, same model -> the honest null for this run
print(f"\nfloor vs REPEAT pass, this path : all {floor_repeat_all:.5f} | test {floor_repeat_test:.5f}")
print(f"floor vs STORED baseline        : all {floor_stored_all:.5f} | test {floor_stored_test:.5f}")
print(f"detector #1 noise floor (test)  : {NOISE_FLOOR:.5f}")

COLAB16_FLOOR = 0.00365        # colab_16's measured floor, stock binning, same substrate/readout
print(f"\ncolab_16 measured floor (stock binning): {COLAB16_FLOOR:.5f}")
if DETERMINISTIC_BINNING and NOISE_FLOOR > 0:
    print(f"this run's floor / colab_16's floor    : {NOISE_FLOOR/COLAB16_FLOOR:.2f}x")
    print("  Both floors are two passes of the SAME frozen base over the SAME cells with the SAME")
    print("  readout; the only difference is the binning tie-break. The shortfall is therefore the")
    print("  share of colab_16's floor attributable to binning randomness rather than to gene")
    print("  subsampling -- the open attribution question colab_18 flagged on colab_16/17.")
    print("  A floor at or near colab_16's would instead mean binning contributed little and gene")
    print("  subsampling carries it, leaving colab_16/17's floor interpretation intact as written.")
print(f"\nstored-vs-repeat gap (test): {floor_stored_test - floor_repeat_test:+.5f} -- this is the size")
print("  of the encoding change between the stored baseline and this path. It is the reason the")
print("  fresh base, not the stored file, is used as the reference for every drift number below.")

# X_BASE_B has done its only job (the repeat-pass floor). X_BASE_A stays -- it is the reference
# every drift number below is measured against, and 6d's substate anchor is computed on it.
del X_BASE_B; gc.collect()
_ram("after base passes")


stored zero-shot labels agree at every aligned row (lineage, substate, apoe_carrier, study_id, donor_id)
  checkpoint parameters loaded: 162 | dropped: 1 (flag_encoder.weight)
  [base reload] all required parameter groups loaded from the checkpoint


base pass A:   0%|          | 0/2228 [00:00<?, ?it/s]

base pass B:   0%|          | 0/2228 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ace8223a980>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ace8223a980>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16


floor vs REPEAT pass, this path : all 0.00225 | test 0.00220
floor vs STORED baseline        : all 0.02701 | test 0.02764
detector #1 noise floor (test)  : 0.00220

colab_16 measured floor (stock binning): 0.00365
this run's floor / colab_16's floor    : 0.60x
  Both floors are two passes of the SAME frozen base over the SAME cells with the SAME
  readout; the only difference is the binning tie-break. The shortfall is therefore the
  share of colab_16's floor attributable to binning randomness rather than to gene
  subsampling -- the open attribution question colab_18 flagged on colab_16/17.
  A floor at or near colab_16's would instead mean binning contributed little and gene
  subsampling carries it, leaving colab_16/17's floor interpretation intact as written.

stored-vs-repeat gap (test): +0.02544 -- this is the size
  of the encoding change between the stored baseline and this path. It is the reason the
  fresh base, not the stored file, is used as the reference for every drift n

> **Interpretation — the measured floor drops to 0.60x of colab_16's, isolating how much of that project's noise floor came from binning tie-break randomness versus gene subsampling -- resolving colab_18's open attribution question (5b).**
>
> Row alignment is confirmed first: the stored zero-shot labels (lineage, substate, apoe_carrier, study_id, donor_id) agree at every row against this session's freshly loaded substrate, so cell-index-based realignment across files is trustworthy for everything that follows. The frozen base then reloads cleanly (162 parameter groups, the same one dropped tensor as every other load).
>
> Two repeat embeddings of the frozen base, both under the deterministic-binning patch, differ by a median 0.00220 on the held-out test cells (0.00225 pooled across all cells) -- this is detector #1's noise floor for this run: the amount two embeddings differ when nothing except gene-subsampling randomness has changed. Against colab_16's own measured floor under the library's stock (non-deterministic) binning, 0.00365, this run's floor is 0.60x as large. Because both floors are two passes of the identical frozen base over the identical cells through the identical readout -- the only thing that differs between them is the binning tie-break rule -- that 0.60x ratio is a direct measurement of how much of colab_16/17's originally reported floor was binning-tie-break noise rather than gene-subsampling noise: roughly 40% of it, with the remaining 60% (this run's floor) attributable to gene subsampling alone. That split treats the two noise sources as additive in cosine-distance units, which holds only to the extent that they are independent and small (`1 - cos(theta) ~ theta^2/2`, so squared displacements -- and hence cosine distances -- add approximately); read it as a first-order attribution, not an exact decomposition. This resolves the mechanism-attribution question colab_18 left open when it first found the binning bug live on the Kang PBMC probe.
>
> Separately, the frozen base compared against colab_10's originally *stored* zero-shot embedding (not a fresh pass on this path) differs by 0.02764 on test cells -- over 12x the floor -- because the stored file was produced under stock binning on a different code path entirely. That stored-vs-repeat gap is exactly why every drift number in this notebook is measured against a freshly re-embedded base on this run's own deterministic path, never against colab_10's stored file: comparing a CPT checkpoint to the wrong-encoding stored baseline would conflate real adaptation drift with this ca. 0.028 encoding-change artifact. The harmless multiprocessing `AssertionError` traceback beneath these numbers is a known cosmetic cleanup-order issue in PyTorch's DataLoader worker teardown, not a computation error -- it fires after the relevant tensors are already returned.


### 5c — Embed the three per-study checkpoints, and colab_16's aggregated adapter on the same path

Each adapter is reloaded onto a fresh base and **merged** before embedding, then the merge is
asserted to have left no adapter machinery in the graph — a `merge_and_unload()` that silently
half-applied would produce a plausible-looking embedding with no error.

The aggregated adapter is re-embedded here, rather than read from colab_16's stored file, so that
§6b's per-study-vs-aggregated comparison differs only in *regime*. Reusing colab_16's stored numbers
would compare per-study checkpoints measured on the patched path against an aggregated checkpoint
measured on the stock path, and the encoding difference would sit inside the comparison. colab_16's
stored values are still printed alongside, as the historical record they are.


In [13]:

from peft import PeftModel

def embed_adapter(adapter_dir, tag):
    """Fresh base + reloaded adapter, merged, embedded over the full substrate."""
    assert os.path.exists(adapter_dir), f"missing adapter {adapter_dir}"
    m = build_scgpt(model_configs, vocab)
    _k2, _ = load_checkpoint_verbose(m, os.path.join(MODEL_DIR, "best_model.pt"), quiet=True)
    assert_required_loaded(m, _k2, tag)
    pm = PeftModel.from_pretrained(m, adapter_dir)
    merged = pm.merge_and_unload()
    leftover = [n for n, mod in merged.named_modules() if hasattr(mod, "lora_A")]
    assert not leftover, f"[{tag}] LoRA layers survived merge_and_unload: {leftover[:5]}"
    assert isinstance(merged.transformer_encoder.layers[0].self_attn, torch.nn.MultiheadAttention), \
        f"[{tag}] attention module is not a plain MultiheadAttention after merge"
    merged.to(DEVICE)
    X = embed_cells(merged, full_ds, desc=f"{tag} embedding")
    assert X.shape == X_BASE_A.shape, f"[{tag}] embedding shape differs from the base pass"
    assert np.isfinite(X).all(), f"[{tag}] embedding contains non-finite values"
    del m, pm, merged; gc.collect(); torch.cuda.empty_cache()
    return X

X_STUDY = {}
for s in STUDIES:
    X_STUDY[s] = embed_adapter(
        os.path.join(DRIVE_ROOT, "scgpt", f"cpt_per_study_{SLUG[s]}_{RUN_TAG}_adapter{SUFFIX}"), s)
    _ram(f"after {s} embedding")

X_AGG = None
AGG_ADAPTER = os.path.join(DRIVE_ROOT, "scgpt", f"cpt_aggregated_{RUN_TAG}_adapter")
if REEMBED_AGGREGATED:
    X_AGG = embed_adapter(AGG_ADAPTER, "aggregated (colab_16)")
    _ram("after aggregated re-embed")
else:
    print("REEMBED_AGGREGATED=False -- 6b will fall back to colab_16's stored numbers, which were")
    print("  measured on the stock-binning path; the comparison then confounds regime with encoding.")

_c16 = audit_prior.get("scgpt_cpt_aggregated", {})
_c16_d1 = _c16.get("detector_1", {})
print("\ncolab_16 stored aggregated numbers (historical record, stock-binning path):")
print(f"  drift_all {_c16_d1.get('drift_all')} | drift_test {_c16_d1.get('drift_test')} "
      f"| floor {_c16_d1.get('noise_floor_measured')}")


  [SEA-AD] all required parameter groups loaded from the checkpoint


SEA-AD embedding:   0%|          | 0/2228 [00:00<?, ?it/s]

[RAM] after SEA-AD embedding      :  10.0 / 179.4 GB (6%)
  [Li2025] all required parameter groups loaded from the checkpoint


Li2025 embedding:   0%|          | 0/2228 [00:00<?, ?it/s]

[RAM] after Li2025 embedding      :  10.3 / 179.4 GB (7%)
  [Haney2024] all required parameter groups loaded from the checkpoint


Haney2024 embedding:   0%|          | 0/2228 [00:00<?, ?it/s]

[RAM] after Haney2024 embedding   :  10.6 / 179.4 GB (7%)
  [aggregated (colab_16)] all required parameter groups loaded from the checkpoint


aggregated (colab_16) embedding:   0%|          | 0/2228 [00:00<?, ?it/s]

[RAM] after aggregated re-embed   :  10.9 / 179.4 GB (7%)

colab_16 stored aggregated numbers (historical record, stock-binning path):
  drift_all 0.08090221881866455 | drift_test 0.08115410804748535 | floor 0.0036469697952270508


> **Interpretation — all four checkpoints (three per-study plus the aggregated re-embed) load and embed cleanly on the same deterministic path (5c).**
>
> Each of the three per-study adapters, plus colab_16's aggregated adapter, is reloaded onto a freshly constructed base and merged before embedding -- each reload reports the checkpoint's parameter groups loading successfully, and RAM climbs by about 0.3 GB per pass (10.0 to 10.9 of 179.4 GB) -- which is exactly the size of one 142,588 x 512 float32 embedding matrix, i.e. the four matrices *are* all kept resident, deliberately: 6a, 6b and 6c compare them against each other and against the base pass. colab_16's aggregated adapter is re-embedded here from its saved adapter file rather than read from that notebook's stored output, specifically so 6b's per-study-vs-aggregated comparison uses embeddings produced on the identical deterministic-binning path as the three new per-study checkpoints -- comparing against colab_16's stored numbers directly (drift_all 0.08090, drift_test 0.08115, floor 0.00365) would reintroduce the same stock-vs-deterministic encoding mismatch 5b's stored-vs-repeat gap just quantified. Those stored numbers are printed here purely as a historical reference point, not as a value anything downstream computes against.

## 6 — Detector #1



### 6a — Per-study drift, gated on the measured floor

Each checkpoint gets two drift figures against the fresh base reference, and the distinction between
them is the one colab_14 had to learn the hard way:

- **`drift_test`** — median per-cell cosine distance over *that study's own held-out test cells*.
  This is the right number for judging one checkpoint against its own noise floor, and the wrong
  number for comparing checkpoints to each other, because each is measured on a different cell
  population. colab_14 showed a single fixed checkpoint moves 2.10x on this metric depending only on
  which study's test cells are read.
- **`drift_all`** — median over all cells in the substrate, identical population for all three. This
  is the only population-matched comparison here and it is what §6b reads.

By contract detector #1 is **reporting-only**: there is no defensible fixed pass bar, so nothing here
is scored against one. `REAL` means only "above the measured floor, therefore interpretable" — it is
not a result.


In [14]:

DETECTOR1 = {}
rows = []
for s in STUDIES:
    cos = per_cell_cosine(X_BASE_A, X_STUDY[s])
    d_all  = 1.0 - float(np.median(cos))
    m_test = test_mask & (study_vals == s)
    assert m_test.any(), f"{s}: no held-out test cells -- cannot gate this checkpoint"
    d_test = 1.0 - float(np.median(cos[m_test]))
    by_lin = {}
    for lin in sorted(np.unique(lineage_vals)):
        m = m_test & (lineage_vals == lin)
        if m.sum() > 0:
            by_lin[lin] = 1.0 - float(np.median(cos[m]))
    inert = d_test <= NOISE_FLOOR
    DETECTOR1[s] = {"drift_all": d_all, "drift_test": d_test,
                    "drift_test_by_lineage": by_lin,
                    "n_test_cells": int(m_test.sum()),
                    "noise_floor_measured": NOISE_FLOOR,
                    "drift_over_floor": d_test / NOISE_FLOOR if NOISE_FLOOR > 0 else float("inf"),
                    "inert": bool(inert), "verdict": "INERT" if inert else "REAL"}
    rows.append({"study": s, "n_test": int(m_test.sum()), "drift_all": round(d_all, 5),
                 "drift_test": round(d_test, 5),
                 "x_floor": round(d_test / NOISE_FLOOR, 2) if NOISE_FLOOR > 0 else None,
                 "verdict": DETECTOR1[s]["verdict"],
                 **{f"drift_{k[:5]}": round(v, 5) for k, v in by_lin.items()}})

if X_AGG is not None:
    cos_a = per_cell_cosine(X_BASE_A, X_AGG)
    AGG_DRIFT_ALL  = 1.0 - float(np.median(cos_a))
    AGG_DRIFT_TEST = 1.0 - float(np.median(cos_a[test_mask]))
    AGG_DRIFT_TEST_BY_STUDY = {s: 1.0 - float(np.median(cos_a[test_mask & (study_vals == s)]))
                               for s in STUDIES}
else:
    AGG_DRIFT_ALL = AGG_DRIFT_TEST = None
    AGG_DRIFT_TEST_BY_STUDY = {}

print(pd.DataFrame(rows).set_index("study").to_string())
print(f"\nmeasured noise floor (held-out test, this path): {NOISE_FLOOR:.5f}")
if X_AGG is not None:
    print(f"aggregated checkpoint, SAME path: drift_all {AGG_DRIFT_ALL:.5f} | drift_test {AGG_DRIFT_TEST:.5f}")
print("\nDetector #1 is reporting-only by contract -- no fixed pass bar is defensible. 'REAL' means")
print("the checkpoint is above the measured floor and therefore interpretable, nothing more.")


           n_test  drift_all  drift_test  x_floor verdict  drift_astro  drift_micro
study                                                                              
SEA-AD      13505    0.04632     0.05336    24.28    REAL      0.05259      0.05461
Li2025       7136    0.04097     0.03541    16.11    REAL      0.03298      0.03866
Haney2024    3160    0.04497     0.04441    20.21    REAL      0.04398      0.04590

measured noise floor (held-out test, this path): 0.00220
aggregated checkpoint, SAME path: drift_all 0.05315 | drift_test 0.05306

Detector #1 is reporting-only by contract -- no fixed pass bar is defensible. 'REAL' means
the checkpoint is above the measured floor and therefore interpretable, nothing more.


> **Interpretation — all three per-study checkpoints register as REAL, 16-24x the measured floor -- the same qualitative result colab_14 found for Geneformer's per-study regime (6a).**
>
> Each per-study checkpoint's drift is measured two ways against the fresh base: `drift_all` (population-matched, all 142,588 cells, the number 6b and 6c use for cross-checkpoint comparison) and `drift_test` (that study's own held-out test cells only, the number judged against the floor). All three clear the measured 0.00220 floor by a wide margin -- SEA-AD 24.28x, Haney2024 20.21x, Li2025 16.11x -- so "REAL" here means each checkpoint's embedding differs from the frozen base by far more than repeat-embedding noise alone could produce, nothing stronger; detector #1 is reporting-only by this project's standing contract, with no fixed pass/fail bar attached to these ratios. The re-embedded aggregated checkpoint (colab_16's adapter, on this run's deterministic path) reads drift_all 0.05315 -- substantially *lower* than its own stored value of 0.08090 (0.66x), not close to it. That direction is what 5b's stored-vs-repath comparison leads one to expect rather than a new discrepancy: the stored number was measured on the stock-binning path, where part of the apparent drift was binning randomness the deterministic path no longer contributes. It is the same-path 0.05315, never the stored 0.08090, that any per-study-vs-aggregated statement below has to be read against. Per-lineage breakdown is also carried here (drift_astro, drift_micro per study) for 6d's substate-reference comparison.


### 6b — The pre-registered test: does the per-study drift spread survive the context cap?

Everything from §3b and §6a comes together here, in the order fixed before the run:

1. **Premise check first.** §3b's flattening factor decides whether this test discriminates at all.
   If the premise failed, the spread below is reported and the prediction is left unresolved — not
   reinterpreted to fit.
2. **Population-matched spread.** `drift_all` for the three checkpoints, max/min, against
   Geneformer's 1.81x. This is the headline.
3. **Rank check.** Does the drift ordering follow pre-cap detected-gene counts, post-cap effective
   length, or step count? colab_14's evidence for the length explanation was a rank match on length
   and a rank mismatch on steps; the same two comparisons are made here.
4. **The population artifact, quantified in-run.** The aggregated checkpoint — one fixed set of
   weights — is sliced onto each study's test cells. Any spread there is caused purely by which cells
   were read, and it is the floor of population-driven variation any `drift_test` comparison
   inherits before a single real adapter difference is added.

The step-count confound (SEA-AD ca. 4.1x Haney2024's updates) is not removed by any of this and is
restated with the verdict rather than after it.


In [15]:

_d_all = pd.Series({s: DETECTOR1[s]["drift_all"] for s in STUDIES})
SPREAD = float(_d_all.max() / _d_all.min()) if _d_all.min() > 0 else float("inf")

print("1) PREMISE (from 3b)")
print(f"   pre-cap median spread {PRE_CAP_SPREAD:.2f}x (need >= {MIN_PRE_CAP_SPREAD}x) -> "
      f"post-cap spread {POST_CAP_SPREAD:.2f}x (need <= {MAX_POST_CAP_SPREAD}x) | "
      f"premise {'HOLDS' if PREMISE_OK else 'WEAK'}")

print("\n2) POPULATION-MATCHED DRIFT SPREAD (all 142,588 cells, identical for all three)")
for s in STUDIES:
    print(f"   {s:12} drift_all {DETECTOR1[s]['drift_all']:.5f}")
print(f"   spread: {_d_all.min():.5f} - {_d_all.max():.5f} = {SPREAD:.2f}x")
print(f"   Geneformer (colab_14, same regime, same slices): {GF_REFERENCE['drift_spread']:.2f}x")

print("\n3) RANK CHECK -- what does the drift ordering track?")
def _rank(d, tie_tol=1e-9):
    """Descending rank; returns None if every value is within tie_tol of the others (uninformative)."""
    vals = list(d.values())
    if (max(vals) - min(vals)) <= tie_tol:
        return None
    return [k for k, _ in sorted(d.items(), key=lambda kv: -kv[1])]

def _fmt(order):
    return ' > '.join(order) if order is not None else "tied -- uninformative"

def _match(order, ref):
    if order is None or ref is None:
        return "tied -- uninformative"
    return "MATCH" if order == ref else "no match"

_order_drift = _rank({s: DETECTOR1[s]["drift_all"] for s in STUDIES})
_order_pre   = _rank({s: GEOMETRY[s]["pre_cap_median_genes"] for s in STUDIES}, tie_tol=1.0)
_order_post  = _rank({s: GEOMETRY[s]["post_cap_median_len"] for s in STUDIES}, tie_tol=1.0)
_order_steps = _rank({s: STEP_BUDGET[s]["max_steps"] for s in STUDIES}, tie_tol=1.0)
print(f"   drift_all (desc)        : {_fmt(_order_drift)}")
print(f"   pre-cap genes (desc)    : {_fmt(_order_pre):30} {_match(_order_pre, _order_drift)}")
print(f"   post-cap length (desc)  : {_fmt(_order_post):30} {_match(_order_post, _order_drift)}")
print(f"   training steps (desc)   : {_fmt(_order_steps):30} {_match(_order_steps, _order_drift)}")
print("   n=3: a rank match here is weak evidence -- there are only 6 possible orderings, so one")
print("   match arises by chance ca. 17% of the time. Read it as consistency, not confirmation.")
print("   NOTE: training steps are a deterministic function of n_train (fixed epoch target), so a")
print("   steps 'MATCH' is really an n_train-size match, not an independent confound check.")

print("\n4) POPULATION ARTIFACT (one fixed checkpoint, three cell populations)")
if AGG_DRIFT_TEST_BY_STUDY:
    _a = pd.Series(AGG_DRIFT_TEST_BY_STUDY)
    for s in STUDIES:
        print(f"   {s:12} aggregated drift_test on this study's cells: {_a[s]:.5f}")
    print(f"   spread {_a.min():.5f} - {_a.max():.5f} = {_a.max()/_a.min():.2f}x, caused ENTIRELY by")
    print("   which cells were read. Any drift_test comparison inherits at least this much.")
else:
    print("   not available (REEMBED_AGGREGATED=False)")

print("\n" + "="*78)
if not PREMISE_OK:
    PREDICTION_VERDICT = "UNRESOLVED -- premise failed"
    print("PREDICTION UNRESOLVED: 3b's premise did not hold, so effective input length still varies")
    print("across studies and the spread above cannot discriminate the length explanation from a")
    print("study-specific one.")
elif SPREAD < GF_REFERENCE["drift_spread"] * 0.7:
    PREDICTION_VERDICT = "CONSISTENT with the length explanation"
    print(f"CONSISTENT WITH THE LENGTH EXPLANATION: spread collapsed to {SPREAD:.2f}x from Geneformer's")
    print(f"{GF_REFERENCE['drift_spread']:.2f}x once the context cap flattened the length lever "
          f"({FLATTENING:.2f}x).")
elif SPREAD >= GF_REFERENCE["drift_spread"]:
    PREDICTION_VERDICT = "AGAINST the length explanation"
    print(f"AGAINST THE LENGTH EXPLANATION: spread survives at {SPREAD:.2f}x despite the cap flattening")
    print(f"length {FLATTENING:.2f}x -- something study-specific, not sequence length, is doing the work.")
else:
    PREDICTION_VERDICT = "INTERMEDIATE -- neither branch cleanly"
    print(f"INTERMEDIATE: spread {SPREAD:.2f}x sits between a collapse and a survival relative to")
    print(f"Geneformer's {GF_REFERENCE['drift_spread']:.2f}x. Neither branch is cleanly supported.")
print("\nCONFOUND, unremoved and restated with the verdict: epoch matching equalizes per-cell")
print(f"exposure, not optimisation budget -- SEA-AD trained {STEP_BUDGET['SEA-AD']['max_steps']} steps vs "
      f"Haney2024's {STEP_BUDGET['Haney2024']['max_steps']} ({STEP_BUDGET['SEA-AD']['max_steps']/STEP_BUDGET['Haney2024']['max_steps']:.1f}x).")
print("N=1 per study, three points, no seed replicates. This can support or embarrass the length")
print("explanation; it cannot settle it.")
print("="*78)


1) PREMISE (from 3b)
   pre-cap median spread 1.78x (need >= 1.3x) -> post-cap spread 1.00x (need <= 1.05x) | premise HOLDS

2) POPULATION-MATCHED DRIFT SPREAD (all 142,588 cells, identical for all three)
   SEA-AD       drift_all 0.04632
   Li2025       drift_all 0.04097
   Haney2024    drift_all 0.04497
   spread: 0.04097 - 0.04632 = 1.13x
   Geneformer (colab_14, same regime, same slices): 1.81x

3) RANK CHECK -- what does the drift ordering track?
   drift_all (desc)        : SEA-AD > Haney2024 > Li2025
   pre-cap genes (desc)    : SEA-AD > Haney2024 > Li2025    MATCH
   post-cap length (desc)  : tied -- uninformative          tied -- uninformative
   training steps (desc)   : SEA-AD > Li2025 > Haney2024    no match
   n=3: a rank match here is weak evidence -- there are only 6 possible orderings, so one
   match arises by chance ca. 17% of the time. Read it as consistency, not confirmation.
   NOTE: training steps are a deterministic function of n_train (fixed epoch target), so a


> **Interpretation — the pre-registered prediction is supported: per-study drift spread collapses from Geneformer's 1.81x to 1.13x once the context cap flattens the length lever -- but a same-checkpoint population-artifact control shows population alone can produce a larger spread (1.38x) than the checkpoint effect itself, so this is read as consistent-with, not confirmed (6b).**
>
> Four things run in the fixed order this test was pre-registered in. First, the premise from 3b is restated and re-confirmed: pre-cap median-length spread was 1.78x (above the 1.3x bar for "a real lever"), and it collapsed to 1.00x post-cap (below the 1.05x tolerance for "flattened") -- premise holds, so this test is licensed to speak to colab_14's length hypothesis.
>
> Second, the actual result: population-matched `drift_all` for the three per-study checkpoints -- SEA-AD 0.04632, Haney2024 0.04497, Li2025 0.04097 -- spans 0.04097 to 0.04632, a 1.13x spread. Geneformer's equivalent per-study spread (colab_14, same donor slices, same regime) was 1.81x. That 1.81x-to-1.13x collapse is the headline finding: it is the direction and rough magnitude colab_14's length explanation predicts once scGPT's 1,200-token cap removes most of the length variation the two architectures would otherwise both be exposed to.
>
> Third, a rank check for consistency (not confirmation, since n=3 gives only 6 possible orderings and one match arises by chance ca. 17% of the time): the drift-magnitude ranking (SEA-AD > Haney2024 > Li2025) matches the pre-cap gene-count ranking exactly, while post-cap length is tied across studies (uninformative by construction) and training-step count ranks differently (SEA-AD > Li2025 > Haney2024, no match) -- though steps are a deterministic function of `n_train` under epoch matching, so this "no match" is really an n_train-size comparison, not an independent confound ruled out.
>
> Fourth, and the reason this result is not read as confirmed: a population-artifact control holds the checkpoint fixed (the re-embedded aggregated adapter) and varies only which study's cells are scored. That alone produces a 1.38x spread (SEA-AD 0.05942, Haney2024 0.04982, Li2025 0.04308; 0.05942/0.04308 = 1.38x) -- larger than the 1.13x checkpoint-identity effect just measured. Since 6a's raw per-study `drift_test` figures (each checkpoint scored on its own study's cells) mix both effects together, and their own spread is only 1.51x (0.05336/0.03541), most of that raw spread is population noise, not checkpoint identity -- which is exactly why `drift_all` (all checkpoints scored on the identical full population) is the number used for the headline comparison rather than `drift_test`.
>
> The confound this cell states plainly rather than removes: epoch matching equalizes per-cell exposure, not optimisation budget -- SEA-AD trained 1,079 steps against Haney2024's 261 (4.1x more updates), and there is no seed-replicate for any of the three checkpoints (N=1 each). The result can support the length explanation or embarrass it; on its own it cannot settle which.


### 6c — Pairwise divergence: did the three adapters move in different directions?

Drift magnitude alone cannot distinguish three checkpoints that moved the same way by different
amounts from three that moved in genuinely different directions. colab_14 found Geneformer's three
58–71° apart — genuinely divergent. The same geometry is computed here.

Cosine-distance quantities do not add or subtract like lengths (`1 - cos(theta) ~ theta^2/2`), so the
angle is recovered explicitly rather than compared as raw distances — the exact error colab_14 had to
correct.


In [16]:

PAIRWISE = {}
print("pairwise distance between per-study checkpoints (median 1 - cosine, all cells):")
for i, a in enumerate(STUDIES):
    for b in STUDIES[i+1:]:
        d = 1.0 - float(np.median(per_cell_cosine(X_STUDY[a], X_STUDY[b])))
        PAIRWISE[f"{a}|{b}"] = d
        print(f"  {a:12} vs {b:12} {d:.5f}")
print("\nfor scale, each checkpoint's own drift vs the fresh base:")
for s in STUDIES:
    print(f"  {s:12} {DETECTOR1[s]['drift_all']:.5f}")

# Angle between two displacement vectors from a shared origin, via the law of cosines on the
# median-cosine distances. Raw distances do NOT subtract linearly -- this is the colab_14 fix.
print("\nangle between displacement directions (law of cosines, from median cosine distances):")
ANGLES = {}
for i, a in enumerate(STUDIES):
    for b in STUDIES[i+1:]:
        da, db = DETECTOR1[a]["drift_all"], DETECTOR1[b]["drift_all"]
        dab = PAIRWISE[f"{a}|{b}"]
        # |u|^2 = 2*d for unit vectors with cosine distance d; law of cosines on the triangle
        la, lb, lab = math.sqrt(2*da), math.sqrt(2*db), math.sqrt(2*dab)
        if la > 0 and lb > 0:
            cosang = max(-1.0, min(1.0, (la**2 + lb**2 - lab**2) / (2*la*lb)))
            ang = math.degrees(math.acos(cosang))
            ANGLES[f"{a}|{b}"] = round(ang, 1)
            print(f"  {a:12} vs {b:12} {ang:5.1f} deg")
print("\ncolab_14's Geneformer checkpoints sat 58-71 deg apart. Near 0 deg would mean the three")
print("adapters moved the same way by different amounts; near 90 deg means genuinely different")
print("directions. This is a geometric description, not a scored eval.")
print("\nHEURISTIC, stated so it is not read as exact: the three sides fed to the law of cosines are")
print("each a MEDIAN over cells, and three per-cell medians do not in general describe one triangle.")
print("The angle is a summary of the typical geometry, not a quantity computed on a single vector")
print("pair -- fine for 'same direction or not', not for a precise angular claim. Same caveat as")
print("colab_14, which used the same construction.")


pairwise distance between per-study checkpoints (median 1 - cosine, all cells):
  SEA-AD       vs Li2025       0.01271
  SEA-AD       vs Haney2024    0.01746
  Li2025       vs Haney2024    0.00828

for scale, each checkpoint's own drift vs the fresh base:
  SEA-AD       0.04632
  Li2025       0.04097
  Haney2024    0.04497

angle between displacement directions (law of cosines, from median cosine distances):
  SEA-AD       vs Li2025        31.1 deg
  SEA-AD       vs Haney2024     36.0 deg
  Li2025       vs Haney2024     25.2 deg

colab_14's Geneformer checkpoints sat 58-71 deg apart. Near 0 deg would mean the three
adapters moved the same way by different amounts; near 90 deg means genuinely different
directions. This is a geometric description, not a scored eval.

HEURISTIC, stated so it is not read as exact: the three sides fed to the law of cosines are
each a MEDIAN over cells, and three per-cell medians do not in general describe one triangle.
The angle is a summary of the typical 

> **Interpretation — per-study checkpoints sit 25-36 deg apart, far closer in direction than Geneformer's 58-71 deg -- and the angle-derivation formula is verified independently here, resolving the discrepancy flagged after the real run (6c).**
>
> Pairwise cosine distance between checkpoint embeddings (median over all cells): SEA-AD vs Li2025 0.01271, SEA-AD vs Haney2024 0.01746, Li2025 vs Haney2024 0.00828. Combined with each checkpoint's own drift_all against the fresh base (SEA-AD 0.04632, Li2025 0.04097, Haney2024 0.04497) via the law of cosines, the recovered angles are SEA-AD-Li2025 31.1 deg, SEA-AD-Haney2024 36.0 deg, Li2025-Haney2024 25.2 deg -- all far closer to 0 deg (same direction, different magnitude) than to 90 deg (genuinely orthogonal), and much closer together than Geneformer's own per-study checkpoints in colab_14 (58-71 deg apart). Read together with 6b: scGPT's three per-study adapters moved a similar *amount* in a comparatively *similar direction*, whereas Geneformer's moved comparably in magnitude but in more genuinely divergent directions -- a real cross-architecture contrast in adaptation geometry, not just adaptation size.
>
> **Discrepancy check, resolved.** A first manual check against these printed angles, done by applying the standard law-of-cosines formula `cos(theta) = (a^2+b^2-c^2)/(2ab)` directly to the raw drift/distance values as if they were already side lengths, reproduced only ~15 deg/~10 deg rather than the printed 31.1 deg/25.2 deg -- an apparent bug. Re-deriving independently from the notebook's own code (reproduced here for SEA-AD vs Li2025: `da=0.04632, db=0.04097, dab=0.01271`) shows the code does not skip a step that manual check did: for unit-normalized embeddings, Euclidean distance and cosine distance relate as `|u|^2 = 2*d`, so the correct triangle side lengths are `la=sqrt(2*da), lb=sqrt(2*db), lab=sqrt(2*dab)`, not `da, db, dab` themselves. Substituting: `cos(theta) = (2da + 2db - 2dab) / (2*sqrt(2da)*sqrt(2db)) = (da+db-dab) / (2*sqrt(da*db))`. Working that through by hand -- `(0.04632+0.04097-0.01271) / (2*sqrt(0.04632*0.04097)) = 0.07458/0.08713 = 0.8560`, `arccos(0.8560) = 31.13 deg` -- reproduces the printed 31.1 deg exactly, and the same substitution for Li2025-Haney2024 gives `(0.04097+0.04497-0.00828) / (2*sqrt(0.04097*0.04497)) = 0.9046`, `arccos(0.9046) = 25.23 deg`, matching the printed 25.2 deg. The third pair behaves the same way: SEA-AD-Haney2024 works out to 36.02 deg against the printed 36.0, where the unconverted manual version would have given 21.99 deg. The earlier ~15 deg/~10 deg mismatch was therefore an error in the *manual check* (omitting the `sqrt(2d)` cosine-distance-to-Euclidean-distance conversion before applying the law of cosines), not in the notebook's own computation -- the code cell's formula is correct as written and its printed angles are verified independently here. The cell's own printed heuristic still applies regardless: each side is a per-cell median, not one measured vector pair, so the recovered angle is a summary of typical geometry ("same direction or not"), not a precise angular claim -- the same caveat colab_14 attached to its own use of this construction.


### 6d — The magnitude anchor, recomputed on this path

A drift number in cosine units means nothing without a same-space reference for what a *biologically
meaningful* distance looks like. The anchor is the within-donor distance between substate poles —
homeostatic vs activated for microglia, resting vs reactive for astrocytes — computed **within
donor**, because the pooled figure is inflated by the donor and study confound documented in
colab_07/08.

Recomputed here rather than carried over: colab_16's 0.0360 / 0.0526 were measured on the
stock-binning path, and every drift number above is on the patched one.


In [17]:

POLES = {"microglia": ("homeostatic", "activated"), "astrocyte": ("resting", "reactive")}
MIN_CELLS_PER_POLE, MIN_DONORS, PAIR_SAMPLE = 50, 5, 3000
substate_v = glia_v.obs["substate"].astype(str).values
donor_v    = glia_v.obs["donor_id"].astype(str).values

def median_pairwise_cos_dist(Xa, Xb, n=PAIR_SAMPLE, seed=0):
    rng = np.random.default_rng(seed)
    ia = rng.choice(len(Xa), size=min(n, len(Xa)), replace=False)
    ib = rng.choice(len(Xb), size=min(n, len(Xb)), replace=False)
    Ua = Xa[ia] / (np.linalg.norm(Xa[ia], axis=1, keepdims=True) + 1e-12)
    Ub = Xb[ib] / (np.linalg.norm(Xb[ib], axis=1, keepdims=True) + 1e-12)
    return float(1.0 - np.median(Ua @ Ub.T))

SUBSTATE_REF = {}
for lin, (p1, p2) in POLES.items():
    m1_all = (lineage_vals == lin) & (substate_v == p1)
    m2_all = (lineage_vals == lin) & (substate_v == p2)
    if m1_all.sum() == 0 or m2_all.sum() == 0:
        print(f"{lin}: a substate pole is absent -- reference not computed")
        SUBSTATE_REF[lin] = None
        continue
    pooled = median_pairwise_cos_dist(X_BASE_A[m1_all], X_BASE_A[m2_all])
    within = []
    for d in np.unique(donor_v):
        m1, m2 = m1_all & (donor_v == d), m2_all & (donor_v == d)
        if m1.sum() >= MIN_CELLS_PER_POLE and m2.sum() >= MIN_CELLS_PER_POLE:
            within.append(median_pairwise_cos_dist(X_BASE_A[m1], X_BASE_A[m2]))
    print(f"\n{lin}: {p1} vs {p2}")
    print(f"  pooled pairwise cos-dist          : {pooled:.4f}  (n={int(m1_all.sum())}/{int(m2_all.sum())})")
    print(f"  qualifying donors (>={MIN_CELLS_PER_POLE} each pole): {len(within)}")
    if len(within) < MIN_DONORS:
        print(f"  UNSTABLE -- fewer than {MIN_DONORS} donors carry both poles; the pooled figure")
        print("  (confound-inflated) is all there is.")
        SUBSTATE_REF[lin] = {"within_donor": None, "pooled": pooled, "n_donors": len(within)}
        continue
    wd = float(np.median(within))
    SUBSTATE_REF[lin] = {"within_donor": wd, "pooled": pooled, "n_donors": len(within),
                         "iqr": [float(np.quantile(within, .25)), float(np.quantile(within, .75))]}
    print(f"  within-donor pairwise cos-dist    : median {wd:.4f} "
          f"(IQR {SUBSTATE_REF[lin]['iqr'][0]:.4f}-{SUBSTATE_REF[lin]['iqr'][1]:.4f})")

C16_REF = {"microglia": 0.0360, "astrocyte": 0.0526}    # colab_16, stock-binning path
print("\ncolab_16's anchors on the stock path, for comparison:", C16_REF)

print("\nper-study drift as % of the within-donor substate reference (this path):")
PCT_OF_REF = {}
for s in STUDIES:
    PCT_OF_REF[s] = {}
    for lin, ref in SUBSTATE_REF.items():
        if not ref:
            continue
        anchor = ref["within_donor"] if ref["within_donor"] is not None else ref["pooled"]
        kind = "within-donor" if ref["within_donor"] is not None else "POOLED"
        d = DETECTOR1[s]["drift_test_by_lineage"].get(lin)
        if d is None or anchor <= 0:
            continue
        PCT_OF_REF[s][lin] = round(100 * d / anchor, 1)
        print(f"  {s:12} {lin:10} drift {d:.5f} = {PCT_OF_REF[s][lin]:6.1f}% of the {kind} reference ({anchor:.4f})")
print("\ncolab_16's aggregated checkpoint read 234.5% (micro) / 149.3% (astro) on the stock path.")
print("Reported without a pass bar, by contract: detector #1 is diagnostic, and the only defensible")
print("anchors are the measured floor and this reference.")



microglia: homeostatic vs activated
  pooled pairwise cos-dist          : 0.0437  (n=25845/12109)
  qualifying donors (>=50 each pole): 36
  within-donor pairwise cos-dist    : median 0.0288 (IQR 0.0229-0.0356)

astrocyte: resting vs reactive
  pooled pairwise cos-dist          : 0.0506  (n=48147/28465)
  qualifying donors (>=50 each pole): 78
  within-donor pairwise cos-dist    : median 0.0425 (IQR 0.0350-0.0520)

colab_16's anchors on the stock path, for comparison: {'microglia': 0.036, 'astrocyte': 0.0526}

per-study drift as % of the within-donor substate reference (this path):
  SEA-AD       microglia  drift 0.05461 =  189.7% of the within-donor reference (0.0288)
  SEA-AD       astrocyte  drift 0.05259 =  123.8% of the within-donor reference (0.0425)
  Li2025       microglia  drift 0.03866 =  134.3% of the within-donor reference (0.0288)
  Li2025       astrocyte  drift 0.03298 =   77.6% of the within-donor reference (0.0425)
  Haney2024    microglia  drift 0.04590 =  159.4% of t

> **Interpretation — recomputed on this run's own deterministic path, every per-study checkpoint's drift still exceeds its biological substate reference except one: Li2025's astrocyte reading (6d).**
>
> The magnitude anchor -- within-donor cosine distance between substate poles (homeostatic vs activated for microglia, resting vs reactive for astrocytes) -- is recomputed here on this notebook's own deterministic-binning path rather than carried over from colab_16's stock-binning numbers, because 5b/5c already established the two paths produce measurably different embeddings. Microglia: 36 donors qualify (>=50 cells each pole), within-donor median 0.0288 (IQR 0.0229-0.0356) -- ca. 20% below colab_16's stock-path figure of 0.0360. Astrocyte: 78 donors qualify, within-donor median 0.0425 (IQR 0.0350-0.0520), again ca. 19% below colab_16's 0.0526. Both anchors use the within-donor distance specifically, not the pooled figure (0.0437 micro / 0.0506 astro), because the pooled number is inflated by the donor/study confound documented back in colab_07/08's substate assignment.
>
> Against these anchors, per-study drift reads: SEA-AD 189.7% (micro) / 123.8% (astro), Haney2024 159.4% / 103.5%, Li2025 134.3% / 77.6%. Five of six readings exceed 100% -- the CPT-induced drift is larger than the biological distance between substate poles within the same donor -- with Li2025's astrocyte reading the sole exception, sitting at 77.6% of the reference. Every per-study figure here is lower than colab_16's aggregated-checkpoint reading (234.5% micro / 149.3% astro) -- but that comparison is cross-path in both its numerator and its denominator, since colab_16's drift *and* its anchor were measured under stock binning, so on its own it is weak evidence about regime. The same-path version of the same statement is 6a's population-matched `drift_all`: 0.04632 / 0.04497 / 0.04097 for the three per-study checkpoints against 0.05315 for the aggregated adapter re-embedded here -- each per-study checkpoint moves less than the aggregated one, on identical cells under one encoding. A second caveat applies to reading the three studies against *each other* in this table: these percentages are built on `drift_test` (each checkpoint on its own study's held-out cells), so they carry the population artifact 6b measured directly, where checkpoint identity held fixed still produced a 1.38x spread. As with every other detector #1 reading in this project, these percentages are reported for context, not scored against a pass bar -- there is no pre-registered threshold for what fraction of the substate reference a CPT drift "should" occupy.

## 7 — Save + handoff



### 7a — Save the embeddings, append the audit trace, print the commit commands

Three per-study embeddings go to Drive for the downstream eval notebook. The audit entry records the
binning path explicitly — colab_18 flagged, as a real gap, that its own patch was not recorded
anywhere in the trace, so a later reader could not tell which encoding produced the numbers. That is
fixed here: `deterministic_binning` travels with the entry and with every saved file's `uns`.


In [18]:

import shlex

SCGPT_OUT_DIR = os.path.join(DRIVE_ROOT, "scgpt")
os.makedirs(SCGPT_OUT_DIR, exist_ok=True)
OBS_COLS = ["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]

saved = {}
for s in STUDIES:
    a = ad.AnnData(X=X_STUDY[s], obs=glia_v.obs[OBS_COLS].copy())
    a.uns["deterministic_binning"] = bool(DETERMINISTIC_BINNING)
    a.uns["reference"] = "fresh base pass A, same notebook, same binning path"
    p = os.path.join(SCGPT_OUT_DIR, f"glia_scgpt_cpt_per_study_{SLUG[s]}_{RUN_TAG}{SUFFIX}.h5ad")
    a.write_h5ad(p)
    saved[s] = os.path.basename(p)
    print(f"saved {s:12} -> {p} ({os.path.getsize(p)/1e9:.2f} GB)")

BASE_PATH = os.path.join(SCGPT_OUT_DIR, f"glia_scgpt_base_detbin_{RUN_TAG}{SUFFIX}.h5ad")
_b = ad.AnnData(X=X_BASE_A, obs=glia_v.obs[OBS_COLS].copy())
_b.uns["deterministic_binning"] = bool(DETERMINISTIC_BINNING)
_b.write_h5ad(BASE_PATH)
print(f"saved base reference -> {BASE_PATH} ({os.path.getsize(BASE_PATH)/1e9:.2f} GB)")

AGG_EMBEDDING_FILE = None
if X_AGG is not None:
    AGG_PATH = os.path.join(SCGPT_OUT_DIR, f"glia_scgpt_cpt_aggregated_repath_{RUN_TAG}{SUFFIX}.h5ad")
    _agg = ad.AnnData(X=X_AGG, obs=glia_v.obs[OBS_COLS].copy())
    _agg.uns["deterministic_binning"] = bool(DETERMINISTIC_BINNING)
    _agg.uns["reference"] = "colab_16 aggregated adapter, re-embedded on this notebook's path"
    _agg.write_h5ad(AGG_PATH)
    AGG_EMBEDDING_FILE = os.path.basename(AGG_PATH)
    print(f"saved aggregated (re-embedded) -> {AGG_PATH} ({os.path.getsize(AGG_PATH)/1e9:.2f} GB)")
else:
    print("REEMBED_AGGREGATED=False -- no aggregated (re-embedded) file to save")

AUDIT_ENTRY = {
    "status": "computed", "date": TODAY, "regime": "per_study", "fm": "scgpt",
    "training_seed": SEED, "donor_split_seed": REF_SEED,
    "model_dir": os.path.basename(MODEL_DIR), "scgpt_commit": SCGPT_PIN,
    "peft_version": peft.__version__,
    "deterministic_binning": bool(DETERMINISTIC_BINNING),
    "determinism_check_maxabs": DETERMINISM_MAXABS,
    "n_cells": int(glia_v.n_obs), "emb_dim": int(X_BASE_A.shape[1]),
    "max_length": MAX_LENGTH, "vocab_audit": VOCAB_AUDIT,
    "budget_rule": "epoch-matched to the aggregated run",
    "target_epochs": round(TARGET_EPOCHS, 4),
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
             "targets": LORA_TARGETS_REGEX, "trainable_params": int(EXPECTED_TRAINABLE)},
    "train_common": {"grad_accum": GRAD_ACCUM, "batch": PER_DEV_BATCH, "eff_batch": EFF_BATCH,
                     "lr": LEARNING_RATE, "mlm_prob": MLM_PROB,
                     "binning_during_training": "stock (matches colab_16)"},
    "noise_floor_measured": NOISE_FLOOR,
    "noise_floor_vs_stored_baseline": floor_stored_test,
    "colab16_floor_stock_binning": COLAB16_FLOOR,
    "input_geometry_by_study": GEOMETRY,
    "pre_cap_spread": PRE_CAP_SPREAD, "post_cap_spread": POST_CAP_SPREAD,
    "flattening_factor": FLATTENING, "premise_ok": PREMISE_OK,
    "premise_thresholds": {"max_post_cap_spread": MAX_POST_CAP_SPREAD,
                           "min_pre_cap_spread": MIN_PRE_CAP_SPREAD},
    "per_study": {s: {**{k: results[s][k] for k in
                         ("slug", "n_train", "n_val", "max_steps", "epochs", "train_seconds",
                          "train_loss_mean", "eval_loss_first", "eval_loss_final", "eval_curve",
                          "adapter_file")},
                      "detector_1": DETECTOR1[s], "pct_of_substate_ref": PCT_OF_REF.get(s, {}),
                      "embedding_file": saved[s]} for s in STUDIES},
    "aggregated_same_path": ({"drift_all": AGG_DRIFT_ALL, "drift_test": AGG_DRIFT_TEST,
                              "drift_test_by_study": AGG_DRIFT_TEST_BY_STUDY,
                              "adapter": os.path.basename(AGG_ADAPTER),
                              "embedding_file": AGG_EMBEDDING_FILE}
                             if X_AGG is not None else None),
    "drift_all_spread": SPREAD,
    "geneformer_reference": GF_REFERENCE,
    "prediction_verdict": PREDICTION_VERDICT,
    "pairwise_distance": PAIRWISE, "pairwise_angle_deg": ANGLES,
    "substate_reference": SUBSTATE_REF,
    "base_embedding_file": os.path.basename(BASE_PATH),
    "donor_split_file": "outputs/donor_split.json",
}

if SMOKE:
    print("\n[SMOKE] plumbing run -- NOT writing audit_report.json")
else:
    with open(AUDIT_PATH) as f:
        report = json.load(f)
    report["scgpt_cpt_per_study"] = AUDIT_ENTRY
    with open(AUDIT_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print("\naudit trace appended ->", AUDIT_PATH)

    print("\n=== Commit + push (from WSL -- Colab has no git creds) ===")
    files = [os.path.relpath(FREEZE_PATH, REPO_PATH), os.path.relpath(ENV_PATH, REPO_PATH),
             "outputs/audit_report.json"]
    print(f"  cd /mnt/c/Users/micic/ad-glia-fm-prep && git add {' '.join(shlex.quote(f) for f in files)}")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
          + shlex.quote("colab_19: scGPT CPT (per-study regime) + detector #1 on a deterministic-binning path"))
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")


saved SEA-AD       -> /content/drive/MyDrive/ad-glia-fm-prep/scgpt/glia_scgpt_cpt_per_study_seaad_seed0.h5ad (0.30 GB)
saved Li2025       -> /content/drive/MyDrive/ad-glia-fm-prep/scgpt/glia_scgpt_cpt_per_study_li2025_seed0.h5ad (0.30 GB)
saved Haney2024    -> /content/drive/MyDrive/ad-glia-fm-prep/scgpt/glia_scgpt_cpt_per_study_haney2024_seed0.h5ad (0.30 GB)
saved base reference -> /content/drive/MyDrive/ad-glia-fm-prep/scgpt/glia_scgpt_base_detbin_seed0.h5ad (0.30 GB)
saved aggregated (re-embedded) -> /content/drive/MyDrive/ad-glia-fm-prep/scgpt/glia_scgpt_cpt_aggregated_repath_seed0.h5ad (0.30 GB)

audit trace appended -> /content/ad-glia-fm-prep/outputs/audit_report.json

=== Commit + push (from WSL -- Colab has no git creds) ===
  cd /mnt/c/Users/micic/ad-glia-fm-prep && git add outputs/software_versions/colab_19_2026-07-28_pip_freeze.txt outputs/software_versions/colab_19_2026-07-28_env.json outputs/audit_report.json
  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m 'colab

> **Interpretation — five embeddings saved to Drive; the audit trace records the deterministic-binning path explicitly, closing the gap colab_18 flagged in its own record (7a).**
>
> Three per-study CPT embeddings, the base reference on this run's deterministic-binning path, and the re-embedded aggregated checkpoint are each saved as a separate 0.30 GB h5ad file to Drive -- five files, matching 5c's four embedding passes plus the base pass from 5b. The audit entry appended to `outputs/audit_report.json` carries `deterministic_binning` as an explicit field on the entry itself and travels with every saved file's `uns` -- specifically closing the gap colab_18 flagged as a real limitation of colab_16/17's records: those two notebooks' stock-binning encoding was never recorded anywhere retrievable, so a later reader reusing their stored pass files could not tell which tie-break rule produced them. This entry does not have that problem. Nothing is pushed automatically from Colab (no git credentials there); the printed commit/push commands are for the manual WSL-side step this project's standing execution convention requires.


### Carried forward

Three per-study LoRA adapters and three full-substrate embeddings on Drive
(`scgpt/cpt_per_study_{seaad,li2025,haney2024}_seed0_adapter` and the matching
`glia_scgpt_cpt_per_study_*_seed0.h5ad`), plus the base reference embedded on the same
deterministic-binning path (`glia_scgpt_base_detbin_seed0.h5ad`) and a `scgpt_cpt_per_study` audit
block.

**Every downstream comparison must use that saved base reference, not colab_10's stored zero-shot
file.** The two were produced under different binning encodings, and mixing them folds an encoding
change into whatever is being measured.

**Next:** evals #1 and #2 (substate probe, APOE k-NN/silhouette) on the three checkpoints, reusing
the colab_17 machinery, reported per study against the aggregated regime and the base reference —
the scGPT counterpart of colab_15.
